# DSC@UIT 2026 - Task 2 LegalQA - Pipeline v3 (leakage-safe anchor extractive)

This is a new experiment derived from the B/v2 baseline.  The original
notebook remains unchanged.  v3 keeps the official Task 2 corpus and models,
uses no API or outside data, and uses Qwen3.5-2B as a legal-span selector.
The pipeline is configured for two 16GB GPUs.

The measured improvement target is assembly and candidate selection: the
answer is copied from a lossless legal span around the selected Khoan/Dieu,
then a small selector chooses among legal-order candidates.


## S00 · Config

Sửa duy nhất ở cell này. `RUN` bật/tắt từng stage; `FORCE` buộc tính lại dù đã có cache.

In [1]:
import os, sys, io, re, json, time, math, pickle, hashlib, random, unicodedata, gc
from pathlib import Path

SEED = 20260817
random.seed(SEED)

# ------------------------------------------------------------------ paths
ART = Path("/kaggle/working/artifacts")
ART.mkdir(parents=True, exist_ok=True)
SUB = Path("/kaggle/working/submit"); SUB.mkdir(parents=True, exist_ok=True)
FT_DIR = Path("/kaggle/working/reranker_ft_v3")   # định nghĩa ở đây, KHÔNG ở cell S14:
                                               # S17 tham chiếu FT_DIR ngay cả khi bỏ qua S14

def _autodetect():
    """Tìm train.json / public-official.json / thư mục context_*.json.
    Override bằng env DSC_DATA_ROOT khi chạy ngoài Kaggle."""
    roots = [Path(p) for p in os.environ.get("DSC_DATA_ROOT", "").split(os.pathsep) if p]
    roots += [Path("/kaggle/input"), Path("."), Path("..")]
    train = pub = ctx = scoring = None
    for r in roots:
        if not r.exists(): continue
        for p in r.rglob("train.json"):
            if train is None: train = p
        for p in r.rglob("public-official.json"):
            if pub is None: pub = p
        if ctx is None:
            for p in r.rglob("context_*.json"):
                ctx = p.parent; break
        if scoring is None:
            for p in r.rglob("scoring.py"):
                scoring = p.parent; break
    return train, pub, ctx, scoring

P_TRAIN, P_PUBLIC, D_CTX, D_SCORING = _autodetect()
print("train  :", P_TRAIN)
print("public :", P_PUBLIC)
print("ctx dir:", D_CTX)
print("scoring:", D_SCORING)
assert P_TRAIN and P_PUBLIC and D_CTX, "Không tìm thấy dữ liệu — kiểm tra Kaggle Dataset đã attach chưa."

# ------------------------------------------------------------------ models
M_EMB     = "AITeamVN/Vietnamese_Embedding_v2"
M_RERANK  = "AITeamVN/Vietnamese_Reranker"
# M_GEN     = "AITeamVN/Vi-Qwen2-1.5B-RAG"
M_GEN     = "Qwen/Qwen3.5-2B"
PARAM_CAP = 4_000_000_000

# ------------------------------------------------------------------ hparams
CFG = dict(
    # chunking — retrieve nhỏ, đọc lớn
    MAX_CHUNK_WORDS   = 450,
    MIN_CHUNK_WORDS   = 25,
    FALLBACK_WIN      = 400,
    FALLBACK_OVERLAP  = 60,

    # sparse
    BM25_K1 = 1.5, BM25_B = 0.75,
    BM25_NGRAM = (1, 2), BM25_MIN_DF = 3, BM25_MAX_FEAT = 1_500_000,
    CHAR_NGRAM = (4, 5), CHAR_MAX_FEAT = 400_000,
    USE_CHAR_RESCORE = False,   # bật = +~40 phút/8000 query; BM25 1-2gram đã phủ phần lớn lexical

    # retrieval
    TOP_SPARSE = 200, TOP_DENSE = 200, TOP_FUSE = 50, TOP_RERANK = 8,
    RRF_K = 60, CIT_BOOST = 3.0,

    # dense / rerank
    EMB_MAX_LEN = 512, EMB_BATCH = 32,
    RERANK_MAX_LEN = 1536, RERANK_BATCH = 8,

    # assembly / length
    R_GRID = [0.8, 1.0, 1.3, 1.6, 2.0, 2.5, 3.0],
    R_DEFAULT = 2.0,
    HARD_MAX_WORDS = 1400,

    # label generation + finetune
    LABEL_N_TRAIN = 0, LABEL_TOP_K = 40,
    FT_GROUP = 4, FT_EPOCHS = 1, FT_LR = 1e-5, FT_TEMP = 0.1, FT_MAXLEN = 768,
    SELECTOR_N_TRAIN = 7000,
    DEV_FRACTION = 0.15, RSWEEP_N = 1200,

    # oof
    N_FOLDS = 5, OOF_EVAL_N = 1500,
)

RUN = dict(
    S02_scorer=True, S03_data=True, S04_chunk=True, S05_oof=True,
    S06_bm25=True, S07_floor=True,
    S08_dense=True, S09_retrieve=True, S10_rerank=True, S11_headroom=True,
    S12_rsweep=True, S13_labels=True, S14_finetune=True,
    S15_candidates=True, S16_selector=True, S17_submit=True,
    S18_generator=True,           # Qwen3.5 chọn span; assemble() vẫn trích nguyên văn
)
FORCE = set()                     # ví dụ: FORCE = {"chunks", "emb_chunks"}

print("\nartifacts ->", ART)

train  : /kaggle/input/datasets/kitnehi1211/uit-task2-publictest/LegalQA - Public Test/train.json
public : /kaggle/input/datasets/kitnehi1211/uit-task2-publictest/LegalQA - Public Test/public-official.json
ctx dir: /kaggle/input/datasets/kitnehi1211/uit-task2-publictest/LegalQA - Public Test/selected-contexts/selected-contexts
scoring: /kaggle/input/datasets/kitnehi1211/uit-task2-scoring/Scoring-Program-Task-LegalQA

artifacts -> /kaggle/working/artifacts


In [2]:
# ============================ V3 ============================
# Qwen3.5 is post-trained and is used as a selector, not as a free-form writer.
# This preserves verbatim legal spans for the official METEOR scorer.
M_GEN = "Qwen/Qwen3.5-2B"
VARIANT = "B3_QWEN35_2B"
GEN_PARAMS_EXPECT = 2_000_000_000

def vname(n):
    return f"{n}_{VARIANT}"

RUN["S18_generator"] = True
print("VARIANT:", VARIANT)
print("generator:", M_GEN, "(legal-span selector)")
print("FT_DIR:", FT_DIR)


VARIANT: B3_QWEN35_2B
generator: Qwen/Qwen3.5-2B (legal-span selector)
FT_DIR: /kaggle/working/reranker_ft_v3


## S01 · Môi trường, GPU, đếm tham số

Đếm tham số **thật** bằng `sum(p.numel())` — BTC tính cả lớp embedding, và với bge-m3 (vocab 250k)
riêng embedding table đã ~256M nên không được ước lượng bằng tên model.

In [3]:
import subprocess, importlib.metadata
def pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)

# Transformers main passes backend=... to safetensors.safe_open.  Kaggle's
# preinstalled safetensors can be older, so upgrade it before importing
# Transformers (upgrading after the extension is loaded requires a restart).
_safe_probe = [
    sys.executable, "-c",
    "import inspect; from safetensors import safe_open; "
    "raise SystemExit(0 if 'backend' in inspect.signature(safe_open).parameters else 1)",
]
_safe_check = subprocess.run(_safe_probe, capture_output=True, text=True)
if _safe_check.returncode != 0:
    pip("--upgrade", "--no-cache-dir", "safetensors>=0.8.0")
    _safe_check = subprocess.run(_safe_probe, capture_output=True, text=True)
    if _safe_check.returncode != 0:
        raise RuntimeError("safetensors installation does not support backend=...: "
                           + (_safe_check.stderr or _safe_check.stdout))
print("safetensors", importlib.metadata.version("safetensors"), "| backend: supported")

try:
    import rouge_score  # noqa
except ImportError:
    if D_SCORING and (D_SCORING / "rouge_score").exists():
        sys.path.insert(0, str(D_SCORING))          # dùng bản bundled của BTC
    else:
        pip("rouge-score")

for mod, pkg in [("nltk","nltk"), ("sklearn","scikit-learn"), ("scipy","scipy"),
                 ("pandas","pandas"), ("pyarrow","pyarrow"), ("lightgbm","lightgbm"),
                 ("accelerate","accelerate"),
                 ("torchvision","torchvision"), ("PIL","pillow"), ("torch", None)]:
    try:
        __import__(mod)
    except ImportError:
        if pkg: pip(pkg)

# Older Transformers releases already expose AutoModelForMultimodalLM but do
# not register model_type='qwen3_5'.  Probe the actual config mapping in a
# child process so an obsolete Transformers module is never cached here.
_qwen35_probe = [
    sys.executable, "-c",
    "from transformers.models.auto.configuration_auto import CONFIG_MAPPING; "
    "raise SystemExit(0 if 'qwen3_5' in CONFIG_MAPPING else 1)",
]
_probe = subprocess.run(_qwen35_probe, capture_output=True, text=True)
if _probe.returncode != 0:
    # The official Qwen3.5 model card requires the latest source build.
    pip("--upgrade", "--no-cache-dir",
        "git+https://github.com/huggingface/transformers.git@main")
    _probe = subprocess.run(_qwen35_probe, capture_output=True, text=True)
    if _probe.returncode != 0:
        raise RuntimeError("Transformers installation still lacks model_type='qwen3_5': "
                           + (_probe.stderr or _probe.stdout))

from transformers import AutoModelForMultimodalLM
from transformers.models.auto.configuration_auto import CONFIG_MAPPING
assert "qwen3_5" in CONFIG_MAPPING, "qwen3_5 config registration missing"
print("transformers", importlib.metadata.version("transformers"),
      "| qwen3_5: supported")

import numpy as np, pandas as pd, torch, nltk
for res in ["wordnet", "omw-1.4"]:
    try:
        nltk.data.find(f"corpora/{res}")
    except LookupError:
        nltk.download(res, quiet=True)

os.environ["TOKENIZERS_PARALLELISM"] = "false"
torch.manual_seed(SEED); np.random.seed(SEED)

NGPU = torch.cuda.device_count()
DEVICES = [f"cuda:{i}" for i in range(NGPU)] or ["cpu"]
print("torch", torch.__version__, "| GPUs:", NGPU, DEVICES)
for i in range(NGPU):
    p = torch.cuda.get_device_properties(i)
    print(f"  cuda:{i} {p.name} {p.total_memory/2**30:.1f}GiB sm{p.major}{p.minor}")
# T4 = sm75 -> KHÔNG có bf16, phải dùng fp16
AMP_DTYPE = torch.float16
print("amp dtype:", AMP_DTYPE)
print("cpus:", os.cpu_count())


def count_params(repo_id, kind="auto"):
    """Đếm tham số thật, tải lên CPU rồi giải phóng ngay."""
    from transformers import AutoModel, AutoModelForSequenceClassification, AutoModelForMultimodalLM
    cls = {"auto": AutoModel, "seq": AutoModelForSequenceClassification,
           "lm": AutoModelForMultimodalLM}[kind]
    kw = dict(torch_dtype=torch.float16, trust_remote_code=True)
    if kind == "seq": kw["num_labels"] = 1
    m = cls.from_pretrained(repo_id, **kw)
    n = sum(p.numel() for p in m.parameters())
    del m; gc.collect()
    return n


def param_budget(include_gen=True):
    rows = [("embedding", M_EMB, count_params(M_EMB, "auto")),
            ("reranker",  M_RERANK, count_params(M_RERANK, "seq"))]
    if include_gen:
        rows.append(("generator", M_GEN, count_params(M_GEN, "lm")))
    tot = sum(r[2] for r in rows)
    print(f"{'role':<11}{'model':<42}{'params':>14}")
    for r, m, n in rows: print(f"{r:<11}{m:<42}{n:>14,}")
    print(f"{'TOTAL':<53}{tot:>14,}   cap {PARAM_CAP:,}   headroom {PARAM_CAP-tot:,}")
    assert tot < PARAM_CAP, "VƯỢT TRẦN 4B — không được nộp cấu hình này"
    json.dump({"rows": [[r, m, n] for r, m, n in rows], "total": tot},
              open(ART / "param_budget.json", "w"), indent=2)
    return tot

# Chạy 1 lần rồi comment lại (tốn ~4 phút tải weights):
# param_budget(include_gen=RUN["S18_generator"])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 69.0 MB/s eta 0:00:00
safetensors 0.8.0 | backend: supported
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 192.5 MB/s eta 0:00:00
transformers 5.17.0.dev0 | qwen3_5: supported
torch 2.10.0+cu128 | GPUs: 2 ['cuda:0', 'cuda:1']
  cuda:0 Tesla T4 14.6GiB sm75
  cuda:1 Tesla T4 14.6GiB sm75
amp dtype: torch.float16
cpus: 4


In [4]:
def param_budget_variant():
    """Đếm tham số thật cho biến thể này. Reranker của A là cross-encoder (seq-cls),
    generator là causal LM."""
    rows = [("embedding", M_EMB, count_params(M_EMB, "auto")),
            ("reranker",  M_RERANK, count_params(M_RERANK, "seq"))]
    if RUN.get("S18_generator", False):
        rows.append(("generator", M_GEN, count_params(M_GEN, "lm")))
    tot = sum(r[2] for r in rows)
    print(f"{'role':<11}{'model':<52}{'params':>14}")
    for r, m, n in rows: print(f"{r:<11}{m:<52}{n:>14,}")
    print(f"{'TOTAL':<63}{tot:>14,}   cap {PARAM_CAP:,}   headroom {PARAM_CAP-tot:,}")
    assert tot < PARAM_CAP, "VƯỢT TRẦN 4B"
    json.dump({"variant": VARIANT, "rows": [[r, m, n] for r, m, n in rows], "total": tot},
              open(ART / f"param_budget_{VARIANT}.json", "w"), indent=2)
    return tot

# param_budget_variant()      # bỏ comment, chạy 1 lần (tải weights ~5 phút)

## S01b · Cache helper

Mọi stage đi qua `cache()`. Cắt session bất kỳ lúc nào rồi chạy lại từ đầu → các stage đã xong trả về ngay.

In [5]:
import numpy as np, pandas as pd

def _p(name, fmt): return ART / f"{name}.{fmt}"

def _save(obj, path, fmt):
    if fmt == "npy":       np.save(path, obj)
    elif fmt == "parquet": obj.to_parquet(path, index=False)
    elif fmt == "json":    json.dump(obj, open(path, "w", encoding="utf-8"), ensure_ascii=False)
    else:                  pickle.dump(obj, open(path, "wb"), protocol=4)

def _load(path, fmt):
    if fmt == "npy":       return np.load(path, allow_pickle=False)
    if fmt == "parquet":   return pd.read_parquet(path)
    if fmt == "json":      return json.load(open(path, encoding="utf-8"))
    return pickle.load(open(path, "rb"))

def cache(name, fn, fmt="pkl"):
    path = _p(name, fmt)
    if path.exists() and name not in FORCE:
        print(f"[cache] {name}.{fmt}")
        return _load(path, fmt)
    t0 = time.time()
    obj = fn()
    _save(obj, path, fmt)
    print(f"[built] {name}.{fmt}  {time.time()-t0:.1f}s  {path.stat().st_size/2**20:.1f}MiB")
    return obj

class T:
    def __init__(s, m): s.m = m
    def __enter__(s): s.t = time.time(); print(f"→ {s.m} ..."); return s
    def __exit__(s, *a): print(f"← {s.m}  {time.time()-s.t:.1f}s")

## S02 · Scorer chính thức, bit-for-bit

Sao y `Scoring-Program-Task-LegalQA/scoring.py`. Ba chi tiết **không được sửa**:
`meteor_score([ref.split()], pred.split())` (tách khoảng trắng thuần, dấu câu dính token),
`use_stemmer=False`, và tokenizer mặc định của `rouge_score` (loại ký tự tiếng Việt có dấu).
ROUGE-L ở đây méo về mặt ngôn ngữ nhưng **phải giữ nguyên** để khớp BTC.

In [6]:
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer

_ROUGE = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)

def meteor_1(ref: str, pred: str) -> float:
    return meteor_score([str(ref).split()], str(pred).split())

def rougeL_1(ref: str, pred: str) -> float:
    return _ROUGE.score(str(ref), str(pred))["rougeL"].fmeasure

def score_official(y_true: dict, y_pred: dict) -> dict:
    """y_true: {id: answer_str}; y_pred: {id: {'answer': str}} — đúng shape BTC."""
    yp = {k: v["answer"] for k, v in y_pred.items()}
    if len(yp) != len(y_true):
        raise Exception(f"số key lệch: pred={len(yp)} vs ref={len(y_true)}")
    ids = list(yp.keys())
    r = np.mean([rougeL_1(y_true[k], yp[k]) for k in ids])
    m = np.mean([meteor_1(y_true[k], yp[k]) for k in ids])
    return {"rouge": float(r), "meteor": float(m)}

if RUN["S02_scorer"]:
    s = "Căn cứ khoản 3 Điều 17 Nghị định 90/2017/NĐ-CP quy định như sau: Phạt tiền từ 4.000.000 đồng."
    assert meteor_1(s, s) > 0.99, "identical phải ~1.0"
    print("identical      :", round(meteor_1(s, s), 4))
    print("một nửa        :", round(meteor_1(s, " ".join(s.split()[:len(s.split())//2])), 4))
    print("dài gấp đôi    :", round(meteor_1(s, s + " " + s.replace("17", "18")), 4))
    print("xáo trộn từ    :", round(meteor_1(s, " ".join(random.sample(s.split(), len(s.split())))), 4))
    print("rouge identical:", round(rougeL_1(s, s), 4))

identical      : 0.9999
một nửa        : 0.5166
dài gấp đôi    : 0.8993
xáo trộn từ    : 0.5
rouge identical: 1.0


> Đọc kỹ bốn số trên: *"một nửa"* và *"xáo trộn từ"* chính là hai cách mất điểm mà pipeline này được thiết kế
> để tránh — ngắn quá, và mất tính liên tục. *"dài gấp đôi"* cho thấy over-length gần như miễn phí.

## S03 · Dữ liệu + duplicate gate

Public QID `129859` trùng chính xác train `88019` → copy thẳng answer. Rẻ, chắc, lấy luôn.

In [7]:
def norm_q(s: str) -> str:
    s = unicodedata.normalize("NFC", str(s)).lower()
    s = re.sub(r"[^\w\s]", " ", s, flags=re.UNICODE)
    return re.sub(r"\s+", " ", s).strip()

def load_data():
    tr = json.load(open(P_TRAIN, encoding="utf-8"))
    pb = json.load(open(P_PUBLIC, encoding="utf-8"))
    return tr, pb

TRAIN, PUBLIC = load_data()
TRAIN_IDS  = sorted(TRAIN.keys())
PUBLIC_IDS = sorted(PUBLIC.keys())
print(f"train {len(TRAIN)}  public {len(PUBLIC)}")

gold_words = np.array([len(TRAIN[i]["answer"].split()) for i in TRAIN_IDS])
print(f"gold words: median {np.median(gold_words):.0f}  mean {gold_words.mean():.0f} "
      f"p90 {np.percentile(gold_words,90):.0f}  max {gold_words.max()}")

# ---- duplicate gate
_tr_by_norm = {}
for i in TRAIN_IDS:
    _tr_by_norm.setdefault(norm_q(TRAIN[i]["question"]), []).append(i)

DUP_GATE = {}
for i in PUBLIC_IDS:
    hit = _tr_by_norm.get(norm_q(PUBLIC[i]["question"]))
    if hit:
        DUP_GATE[i] = max(hit, key=lambda t: len(TRAIN[t]["answer"]))
print("public trùng train (exact sau chuẩn hoá):", DUP_GATE)

# nhóm câu trùng trong train -> group id cho OOF (chống leakage)
QGROUP = {}
for gi, (nq, ids) in enumerate(sorted(_tr_by_norm.items())):
    for i in ids: QGROUP[i] = gi
print("train: %d câu -> %d nhóm (%d nhóm có >1 câu)" %
      (len(TRAIN), len(set(QGROUP.values())),
       sum(1 for v in _tr_by_norm.values() if len(v) > 1)))

train 7000  public 1000
gold words: median 312  mean 347 p90 576  max 2435
public trùng train (exact sau chuẩn hoá): {'129859': '88019'}
train: 7000 câu -> 6983 nhóm (16 nhóm có >1 câu)


## S04 · Chunk corpus theo cấu trúc pháp lý

Ưu tiên ranh giới **Điều**. Ba cạm bẫy được xử lý:
1. **Điều xuất hiện hai lần** (mục lục ở đầu + nội dung thật) → giữ bản **dài nhất** theo số Điều.
2. **Điều quá dài** → chia tiếp theo **Khoản**, nhưng lưu `parent_key` để `assemble()` gộp lại nguyên Điều.
3. **1.125 file thiếu `name`** và **20 passage rỗng** → suy `van_ban` từ header passage / slug link, không crash.

`text_raw` giữ **nguyên byte**: không normalize số (`4.000.000`), không sửa dấu câu, không bỏ dòng trắng —
vì METEOR tách `.split()` nên mỗi thay đổi định dạng là mất token khớp.

In [8]:
import glob

DIEU_RE  = re.compile(r"(?m)^[ \t]*(Điều\s+(\d+[a-zđ]?))\s*[.:\-]?[ \t]*(.*)$")
KHOAN_RE = re.compile(r"(?m)^[ \t]*(\d{1,2})[.)]\s")
SO_RE    = re.compile(r"(?m)^[ \t]*Số[ \t]*:?[ \t]*([\w][\w/\.\-]*\d[\w/\.\-]*)")
VB_TYPES = ["Nghị định","Thông tư liên tịch","Thông tư","Quyết định","Nghị quyết",
            "Pháp lệnh","Chỉ thị","Bộ luật","Luật","Công văn","Hướng dẫn","Kế hoạch"]
VB_TYPE_RE = re.compile(r"(?i)\b(" + "|".join(VB_TYPES) + r")\b")
SLUG_MAP = {"nghi-dinh":"Nghị định","thong-tu":"Thông tư","quyet-dinh":"Quyết định",
            "nghi-quyet":"Nghị quyết","phap-lenh":"Pháp lệnh","chi-thi":"Chỉ thị",
            "bo-luat":"Bộ luật","luat":"Luật","cong-van":"Công văn"}

def extract_van_ban(passage: str, name: str, link: str) -> str:
    """-> 'Nghị định 90/2017/NĐ-CP' (best effort, không bao giờ raise)."""
    head = (passage or "")[:4000]
    mno, mty = SO_RE.search(head), VB_TYPE_RE.search(head)
    if mno:
        num = mno.group(1).strip(" .,;")
        ty = mty.group(1) if mty else None
        if ty: return f"{ty[0].upper()+ty[1:].lower()} {num}"
        return num
    slug = (name or "") or (link or "")
    # Mã cơ quan phải khớp CHỮ HOA (case-sensitive). Slug có dạng
    #   Thong-tu-17-2022-TT-BGTVT-sua-doi-Thong-tu-12-2020-...
    # phần mô tả là chữ thường, nên nếu bật (?i) cho cả nhóm này thì nó ăn luôn
    # '-sua-doi-...' rồi .upper() ra 'TT-BGTVT-SUA'. Tách hai bước:
    m = re.search(r"(?i)(" + "|".join(SLUG_MAP) + r")-(\d{1,4})-(\d{4})", slug)
    if m:
        ty, num, yr = SLUG_MAP[m.group(1).lower()], m.group(2), m.group(3)
        mc = re.match(r"-([A-Z]{2,6}(?:-[A-Z]{2,10}){0,2})(?=-[a-z0-9]|-?$)", slug[m.end():])
        return f"{ty} {num}/{yr}/{mc.group(1)}" if mc else f"{ty} {num}/{yr}"
    base = re.sub(r"-\d+$", "", (name or "").replace("-", " ")).strip()
    return base[:120] or f"doc {slug[-30:]}"

def _win_chunks(text, win, ov):
    w = text.split()
    if not w: return []
    out, i = [], 0
    while i < len(w):
        out.append(" ".join(w[i:i+win]))
        if i + win >= len(w): break
        i += win - ov
    return out

def _split_exact(text: str, max_w: int):
    """Chia theo ranh giới DÒNG, KHÔNG overlap, giữ nguyên từng ký tự.
    Bắt buộc dùng bên trong một Điều: `_full_dieu()` ghép các part lại bằng ''.join(),
    nên phép chia phải lossless. `_win_chunks` (split/join + overlap) sẽ làm lặp từ
    và mất newline -> phá vỡ nguyên tắc copy nguyên byte."""
    lines = text.splitlines(keepends=True)
    out, cur, n = [], [], 0
    for ln in lines:
        k = len(ln.split())
        if cur and n + k > max_w:
            out.append("".join(cur)); cur, n = [], 0
        cur.append(ln); n += k
    if cur: out.append("".join(cur))
    return [o for o in out if o.strip()] or [text]

def split_dieu(body: str, max_w: int):
    """Chia một Điều dài theo ranh giới Khoản, gom tới ~max_w từ mỗi mảnh."""
    marks = [m.start() for m in KHOAN_RE.finditer(body)]
    if not marks:
        return _split_exact(body, max_w)
    segs = [body[:marks[0]]] if marks[0] > 0 else []
    for a, b in zip(marks, marks[1:] + [len(body)]):
        segs.append(body[a:b])
    out, cur, n = [], [], 0
    for s in segs:
        k = len(s.split())
        if cur and n + k > max_w:
            out.append("".join(cur)); cur, n = [], 0
        cur.append(s); n += k
    if cur: out.append("".join(cur))
    # HARD CAP: một Khoản đơn lẻ có thể tự nó dài 30k từ (bảng biểu, phụ lục) và lọt qua
    # bộ gom ở trên -> phá vỡ giả định 512 token của encoder. Cắt cửa sổ cho phần dư.
    final = []
    for o in out:
        if not o.strip(): continue
        if len(o.split()) > 2 * max_w:
            final += _split_exact(o, max_w)
        else:
            final.append(o)
    return final

def build_chunks():
    files = sorted(glob.glob(str(D_CTX / "context_*.json")))
    rows, empty = [], 0
    for fi, f in enumerate(files):
        if fi % 1500 == 0: print(f"  {fi}/{len(files)}  rows={len(rows)}")
        d = json.load(open(f, encoding="utf-8"))
        doc_id, passage = d.get("id"), d.get("passage") or ""
        if not passage.strip():
            empty += 1; continue
        van_ban = extract_van_ban(passage, d.get("name") or "", d.get("link") or "")
        ms = list(DIEU_RE.finditer(passage))
        # (1) Điều trùng -> giữ span dài nhất
        best = {}
        for j, m in enumerate(ms):
            no = m.group(2)
            end = ms[j+1].start() if j+1 < len(ms) else len(passage)
            if no not in best or (end - m.start()) > (best[no][1] - best[no][0]):
                best[no] = (m.start(), end, m.group(3).strip())
        if best:
            for no, (a, b, title) in sorted(best.items(), key=lambda kv: kv[1][0]):
                body = passage[a:b]
                if len(body.split()) < CFG["MIN_CHUNK_WORDS"]: continue
                pieces = ([body] if len(body.split()) <= CFG["MAX_CHUNK_WORDS"]
                          else split_dieu(body, CFG["MAX_CHUNK_WORDS"]))
                for pi, pc in enumerate(pieces):
                    rows.append(dict(doc_id=doc_id, van_ban=van_ban, dieu_no=no,
                                     dieu_title=title[:300], part=pi, n_parts=len(pieces),
                                     parent_key=f"{doc_id}|{no}", text_raw=pc,
                                     n_words=len(pc.split()), link=d.get("link") or ""))
        else:  # (3) không nhận diện được cấu trúc -> cửa sổ có overlap
            for pi, pc in enumerate(_win_chunks(passage, CFG["FALLBACK_WIN"], CFG["FALLBACK_OVERLAP"])):
                rows.append(dict(doc_id=doc_id, van_ban=van_ban, dieu_no="", dieu_title="",
                                 part=pi, n_parts=-1, parent_key=f"{doc_id}|W{pi}",
                                 text_raw=pc, n_words=len(pc.split()), link=d.get("link") or ""))
    df = pd.DataFrame(rows)
    df["chunk_id"] = np.arange(len(df), dtype=np.int64)
    print(f"docs={len(files)} empty={empty} chunks={len(df)}")
    return df

if RUN["S04_chunk"]:
    CH = cache("chunks", build_chunks, "parquet")
    print(CH[["doc_id","van_ban","dieu_no","n_words"]].head(8).to_string())
    print("\nn_words:", CH.n_words.describe()[["mean","50%","max"]].to_dict())
    print("chunk có dieu_no:", (CH.dieu_no != "").mean().round(3))
    print("van_ban parse ok:", (~CH.van_ban.str.startswith("doc ")).mean().round(3))
    print("chunk > 2*MAX_CHUNK_WORDS:", int((CH.n_words > 2*CFG["MAX_CHUNK_WORDS"]).sum()))
    CH_TEXT = CH.text_raw.tolist()
    PARENT_ROWS = CH.groupby("parent_key").indices     # parent_key -> row idx (gộp lại nguyên Điều)

    # bảo vệ tính lossless: ghép các part của một Điều phải ra đúng text gốc liền mạch.
    # Nếu assert này vỡ thì `_full_dieu()` đang tạo text lặp/mất newline -> sai toàn bộ nhãn METEOR.
    _multi = [pk for pk, rs in PARENT_ROWS.items() if len(rs) > 1][:200]
    for pk in _multi:
        rs = sorted(PARENT_ROWS[pk], key=lambda r: CH.part.values[r])
        joined = "".join(CH_TEXT[r] for r in rs)
        assert sum(len(CH_TEXT[r]) for r in rs) == len(joined), f"part chồng lấn tại {pk}"
    print(f"lossless check: OK trên {len(_multi)} Điều nhiều part")

  0/8532  rows=0
  1500/8532  rows=44486
  3000/8532  rows=88839
  4500/8532  rows=133233
  6000/8532  rows=176495
  7500/8532  rows=223521
docs=8532 empty=20 chunks=252971
[built] chunks.parquet  98.4s  145.9MiB
   doc_id                             van_ban dieu_no  n_words
0  100050  doc hom-cay-luong-thuc-919415.aspx              400
1  100050  doc hom-cay-luong-thuc-919415.aspx              400
2  100050  doc hom-cay-luong-thuc-919415.aspx              400
3  100050  doc hom-cay-luong-thuc-919415.aspx              400
4  100050  doc hom-cay-luong-thuc-919415.aspx              400
5  100050  doc hom-cay-luong-thuc-919415.aspx              400
6  100050  doc hom-cay-luong-thuc-919415.aspx              400
7  100050  doc hom-cay-luong-thuc-919415.aspx              237

n_words: {'mean': 270.16977044799603, '50%': 267.0, 'max': 1326.0}
chunk có dieu_no: 0.929
van_ban parse ok: 0.945
chunk > 2*MAX_CHUNK_WORDS: 14
lossless check: OK trên 200 Điều nhiều part


## S05 · OOF split (group theo câu hỏi chuẩn hoá)

Train có nhóm câu hỏi trùng nhau; nếu để chúng rơi khác fold thì OOF sẽ lạc quan giả và mọi quyết định
sau đó đều lệch. `GroupKFold` trên `QGROUP`.

In [9]:
from sklearn.model_selection import GroupKFold

def build_oof():
    ids = np.array(TRAIN_IDS)
    grp = np.array([QGROUP[i] for i in ids])
    fold = np.full(len(ids), -1, dtype=np.int8)
    for k, (_, te) in enumerate(GroupKFold(n_splits=CFG["N_FOLDS"]).split(ids, groups=grp)):
        fold[te] = k
    return pd.DataFrame({"qid": ids, "group": grp, "fold": fold})

if RUN["S05_oof"]:
    OOF = cache("oof_split_v3", build_oof, "parquet")
    print(OOF.fold.value_counts().sort_index().to_dict())
    rng = np.random.default_rng(SEED + 31)
    all_groups = np.array(sorted(OOF.group.unique()))
    n_dev_groups = max(1, int(round(len(all_groups) * CFG["DEV_FRACTION"])))
    dev_groups = set(rng.choice(all_groups, size=n_dev_groups, replace=False).tolist())
    DEV_IDS = [str(q) for q in OOF.loc[OOF.group.isin(dev_groups), "qid"].tolist()]
    FIT_IDS = [str(q) for q in OOF.loc[~OOF.group.isin(dev_groups), "qid"].tolist()]
    EVAL_IDS = DEV_IDS
    assert not set(FIT_IDS) & set(DEV_IDS)
    assert set(FIT_IDS) | set(DEV_IDS) == set(TRAIN_IDS)
    print("FIT_IDS:", len(FIT_IDS), "DEV_IDS:", len(DEV_IDS),
          "groups disjoint:", not bool(set(QGROUP[i] for i in FIT_IDS) &
                                          set(QGROUP[i] for i in DEV_IDS)))


[built] oof_split_v3.parquet  0.0s  0.1MiB
{0: 1400, 1: 1400, 2: 1400, 3: 1400, 4: 1400}
FIT_IDS: 5952 DEV_IDS: 1048 groups disjoint: True


## S06 · Sparse: BM25 (1-2gram) + citation parser

BM25 tự cài bằng `scipy` sparse — nhanh hơn `rank_bm25` vài bậc trên 8.000 query × ~125k chunk,
và không thêm dependency. Kênh char `4-5gram` **không** làm index toàn corpus (quá tốn RAM) mà chỉ
**rescore trên tập ứng viên** — bắt lỗi chính tả và biến thể viết số văn bản.

Citation parser tách hai pass độc lập (số Điều / mã văn bản / số khoản) thay vì một regex tổ hợp — dễ vỡ hơn nhiều.

In [10]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import scipy.sparse as sp

def bm25_fit(texts, ngram, min_df, max_feat, k1, b):
    cv = CountVectorizer(ngram_range=ngram, min_df=min_df, max_features=max_feat,
                         lowercase=True, dtype=np.int32)
    X = cv.fit_transform(texts)                       # docs x terms
    N = X.shape[0]
    df = np.bincount(X.indices, minlength=X.shape[1])
    idf = np.log(1.0 + (N - df + 0.5) / (df + 0.5)).astype(np.float32)
    dl = np.asarray(X.sum(1)).ravel().astype(np.float32)
    avg = float(dl.mean())
    Xc = X.tocsr().astype(np.float32)
    rows = np.repeat(np.arange(N), np.diff(Xc.indptr))
    denom = Xc.data + k1 * (1 - b + b * dl[rows] / avg)
    Xc.data = Xc.data * (k1 + 1) / denom              # doc-side BM25 weight
    return dict(cv=cv, W=Xc, idf=idf)

def bm25_query(idx, queries, topk):
    q = idx["cv"].transform(queries).astype(np.float32)      # nq x terms
    q = q.multiply(idx["idf"][None, :]).tocsr()
    S = (q @ idx["W"].T).toarray()                           # nq x ndocs
    top = np.argpartition(-S, topk, axis=1)[:, :topk]
    ord_ = np.take_along_axis(S, top, 1).argsort(1)[:, ::-1]
    top = np.take_along_axis(top, ord_, 1)
    return top, np.take_along_axis(S, top, 1)

# ------------------------------------------------------------ citation
VB_NUM_RE = re.compile(r"(?i)\b(" + "|".join(VB_TYPES) + r")\s+(?:số\s+)?(\d{1,4}\s*/\s*\d{4}(?:\s*/\s*[A-ZĐa-zđ\-]+)?)")
DIEU_NUM_RE = re.compile(r"(?i)\bĐiều\s+(\d+[a-zđ]?)")
KHOAN_NUM_RE = re.compile(r"(?i)\bkhoản\s+(\d+)")

def parse_cit(text: str) -> dict:
    t = str(text)
    vbs = [a[0].upper() + a[1:].lower() + " " + re.sub(r"\s+", "", b)
           for a, b in VB_NUM_RE.findall(t)]
    return dict(van_ban=list(dict.fromkeys(vbs)),
                dieu=list(dict.fromkeys(DIEU_NUM_RE.findall(t))),
                khoan=list(dict.fromkeys(KHOAN_NUM_RE.findall(t))))

if RUN["S06_bm25"]:
    with T("BM25 fit"):
        BM = cache("bm25", lambda: bm25_fit(CH_TEXT, CFG["BM25_NGRAM"], CFG["BM25_MIN_DF"],
                                            CFG["BM25_MAX_FEAT"], CFG["BM25_K1"], CFG["BM25_B"]))
    print("vocab", len(BM["cv"].vocabulary_), "| W nnz", BM["W"].nnz)

    # index tra cứu citation -> chunk row
    VB_DIEU = {}
    for r, (vb, dn) in enumerate(zip(CH.van_ban.values, CH.dieu_no.values)):
        if dn: VB_DIEU.setdefault((vb, str(dn)), []).append(r)
    print("cặp (văn bản, Điều) duy nhất:", len(VB_DIEU))

    q = TRAIN[TRAIN_IDS[0]]["question"]
    print("\nví dụ citation trong question:", parse_cit(q))
    print("ví dụ citation trong answer  :", parse_cit(TRAIN[TRAIN_IDS[0]]["answer"])["van_ban"][:3])
    n_cit = sum(1 for i in TRAIN_IDS if parse_cit(TRAIN[i]["answer"])["van_ban"])
    print(f"answer có mã văn bản: {n_cit}/{len(TRAIN_IDS)}")

→ BM25 fit ...
[built] bm25.pkl  99.3s  544.5MiB
← BM25 fit  99.3s
vocab 647465 | W nnz 68312578
cặp (văn bản, Điều) duy nhất: 146027

ví dụ citation trong question: {'van_ban': [], 'dieu': [], 'khoan': []}
ví dụ citation trong answer  : ['Nghị định 09/2021/NĐ-CP']
answer có mã văn bản: 4097/7000


## S07 · `assemble()` — **nguồn chân lý duy nhất**

Mọi thứ downstream phụ thuộc cell này: nhãn METEOR ở S13, finetune ở S14, candidate ở S15, submission ở S17.
Sửa `assemble()` mà không tính lại nhãn = **train/serve mismatch**, và reranker sẽ tối ưu cho một hàm không tồn tại.

Bốn quy tắc:
- Mở đầu `"Căn cứ ..."` (57% gold dùng mẫu này) — cho ~15–25 token khớp **liền mạch** mỗi câu, gần như miễn phí.
- Body **copy nguyên byte**, gộp sub-chunk về **Điều đầy đủ** (`parent_key`).
- Thiếu so với `L*` → **mở rộng** sang chunk kế tiếp (bất đối xứng 5:1 nghiêng về over-length).
- Vượt `L*` → cắt **theo ranh giới dòng**, không cắt giữa dòng, để giữ `ch/m` thấp.

Không sinh phần `"Trước đây, căn cứ ..."`: chỉ ≤39/7000 gold có, thêm vào là đẩy `r` vô ích.

In [11]:
QTYPE_RULES = [
    ("xu_phat",   r"xử phạt|phạt tiền|mức phạt|bị phạt|vi phạm hành chính"),
    ("mau_bieu",  r"^\s*mẫu\b|biểu mẫu|tờ khai|mẫu số"),
    ("ho_so",     r"hồ sơ|thủ tục|trình tự|các bước|đăng ký|cấp giấy"),
    ("trach_nhiem", r"trách nhiệm|nhiệm vụ|quyền hạn|nghĩa vụ|có quyền"),
    ("dieu_kien", r"điều kiện|tiêu chuẩn|yêu cầu gì|tiêu chí"),
    ("dinh_nghia", r"là gì|được hiểu|khái niệm|định nghĩa"),
    ("muc_huong", r"mức hưởng|trợ cấp|phụ cấp|tiền lương|bao nhiêu tiền|hệ số"),
    ("thoi_han",  r"thời hạn|thời gian|bao lâu|khi nào|mấy ngày"),
    ("tham_quyen", r"thẩm quyền|cơ quan nào|ai có quyền|do ai"),
]

def qtype(q: str) -> str:
    ql = str(q).lower()
    for name, pat in QTYPE_RULES:
        if re.search(pat, ql):
            return name
    return "khac"

def build_len_table():
    d = {}
    for i in TRAIN_IDS:
        d.setdefault(qtype(TRAIN[i]["question"]), []).append(len(TRAIN[i]["answer"].split()))
    tab = {k: float(np.median(v)) for k, v in d.items()}
    tab["__global__"] = float(np.median(gold_words))
    return tab

if RUN["S07_floor"]:
    LEN_TAB = cache("len_table", build_len_table, "json")
    cnt = {}
    for i in TRAIN_IDS:
        cnt[qtype(TRAIN[i]["question"])] = cnt.get(qtype(TRAIN[i]["question"]), 0) + 1
    print(f"{'qtype':<14}{'n':>6}{'median gold words':>20}")
    for k in sorted(cnt, key=lambda x: -cnt[x]):
        print(f"{k:<14}{cnt[k]:>6}{LEN_TAB[k]:>20.0f}")

def q_rstar(question: str) -> float:
    # R_BY_TYPE is learned in S12 from official train answers.  Fall back to
    # the global value when a small/unknown bucket is absent.
    table = globals().get("R_BY_TYPE", {})
    return float(table.get(qtype(question), CFG["R_DEFAULT"]))

def target_words(question: str, r: float | None = None) -> int:
    if r is None:
        r = q_rstar(question)
    base = LEN_TAB.get(qtype(question), LEN_TAB["__global__"])
    return int(min(CFG["HARD_MAX_WORDS"], max(60, round(float(r) * base))))

CH_WORDS = np.asarray([len(t.split()) for t in CH_TEXT], dtype=np.int32)
_PARENT_SORTED = {
    pk: sorted([int(r) for r in rs], key=lambda r: (int(CH.part.values[r]), int(CH.chunk_id.values[r])))
    for pk, rs in PARENT_ROWS.items()
}

def _parent_rows(row_idx: int):
    pk = CH.parent_key.values[row_idx]
    return _PARENT_SORTED.get(pk, [int(row_idx)])

def _full_dieu(row_idx: int) -> str:
    """Backward-compatible full parent text for copy-rate diagnostics."""
    return "".join(CH_TEXT[r] for r in _parent_rows(int(row_idx)))

def _window_containing(rows, anchor_pos: int, max_words: int) -> str:
    """Largest lossless contiguous part window containing the anchor."""
    max_words = max(1, int(max_words))
    weights = np.asarray([CH_WORDS[r] for r in rows], dtype=np.int64)
    if weights[anchor_pos] > max_words:
        return _text_window(CH_TEXT[rows[anchor_pos]], "", max_words)
    prefix = np.concatenate(([0], np.cumsum(weights)))
    best = None
    best_key = None
    for a in range(0, anchor_pos + 1):
        if prefix[anchor_pos + 1] - prefix[a] > max_words:
            continue
        b = int(np.searchsorted(prefix, prefix[a] + max_words, side="right") - 1)
        b = max(anchor_pos + 1, min(len(rows), b))
        n = int(prefix[b] - prefix[a])
        key = (n, -abs((anchor_pos - a) - (b - anchor_pos - 1)))
        if best_key is None or key > best_key:
            best, best_key = (a, b), key
    if best is not None:
        a, b = best
        return "".join(CH_TEXT[r] for r in rows[a:b])
    # A single part may exceed the target.  Prefer a line window around the
    # legal/query anchor before falling back to a prefix.
    return _text_window(CH_TEXT[rows[anchor_pos]], "", max_words)

def _text_window(text: str, question: str, max_words: int) -> str:
    """Select a lossless line window around a citation/query anchor."""
    if len(text.split()) <= max_words:
        return text
    lines = text.splitlines(keepends=True)
    if not lines:
        return _trim(text, max_words)
    wanted = set(parse_cit(question)["khoan"])
    anchor = 0
    if wanted:
        for j, line in enumerate(lines):
            if wanted.intersection(KHOAN_RE.findall(line)):
                anchor = j
                break
    elif question:
        qtok = set(re.findall(r"\w+", question.lower(), flags=re.UNICODE))
        overlaps = [len(qtok.intersection(set(re.findall(r"\w+", line.lower(), flags=re.UNICODE))))
                    for line in lines]
        anchor = int(np.argmax(overlaps)) if overlaps and max(overlaps) > 0 else 0
    weights = np.asarray([len(line.split()) for line in lines], dtype=np.int64)
    prefix = np.concatenate(([0], np.cumsum(weights)))
    best, best_key = None, None
    for a in range(anchor + 1):
        if prefix[anchor + 1] - prefix[a] > max_words:
            continue
        b = int(np.searchsorted(prefix, prefix[a] + max_words, side="right") - 1)
        b = max(anchor + 1, min(len(lines), b))
        n = int(prefix[b] - prefix[a])
        key = (n, -abs((anchor - a) - (b - anchor - 1)))
        if best_key is None or key > best_key:
            best, best_key = (a, b), key
    if best is not None:
        a, b = best
        return "".join(lines[a:b])
    return _trim(lines[anchor], max_words)

def _anchor_body(row_idx: int, question: str, max_words: int) -> str:
    rows = _parent_rows(row_idx)
    if len(rows) == 1:
        return _text_window(CH_TEXT[rows[0]], question, max_words)
    anchor_pos = rows.index(row_idx) if row_idx in rows else 0
    wanted = set(parse_cit(question)["khoan"])
    if wanted:
        for j, r in enumerate(rows):
            if wanted.intersection(KHOAN_RE.findall(CH_TEXT[r])):
                anchor_pos = j
                break
    text = _window_containing(rows, anchor_pos, max_words)
    if len(text.split()) <= max_words:
        return text
    return _text_window(CH_TEXT[rows[anchor_pos]], question, max_words)

def _ordered_representatives(row_idxs):
    reps = []
    seen = set()
    for ri in row_idxs:
        ri = int(ri)
        pk = CH.parent_key.values[ri]
        if pk not in seen:
            seen.add(pk)
            reps.append(ri)
    if len(reps) > 1 and len({CH.doc_id.values[r] for r in reps}) == 1:
        reps.sort(key=lambda r: int(CH.chunk_id.values[r]))
    return reps

def _head(row_idx: int, mode: str = "default") -> str:
    vb = CH.van_ban.values[row_idx]
    dn = CH.dieu_no.values[row_idx]
    ti = CH.dieu_title.values[row_idx]
    if mode == "none":
        return ""
    if dn:
        if mode == "theo":
            return f"Theo Điều {dn} {vb} quy định như sau:"
        if mode == "notitle":
            return f"Căn cứ Điều {dn} {vb} quy định như sau:"
        s = f"Căn cứ Điều {dn} {vb}"
        return s + (f" quy định về {ti.rstrip('.')} như sau:" if ti else " quy định như sau:")
    return f"Căn cứ {vb} quy định như sau:"

def _trim(text: str, target: int) -> str:
    if len(text.split()) <= target:
        return text
    lines, out, n = text.split("\n"), [], 0
    for ln in lines:
        k = len(ln.split())
        if out and n + k > target:
            break
        out.append(ln)
        n += k
    return "\n".join(out) if out else " ".join(text.split()[:target])

def assemble(row_idxs, question: str, r: float | None = None,
            expand: bool = True, head_mode: str = "default") -> str:
    """Anchor-centered, legal-order, extractive answer assembly."""
    r = q_rstar(question) if r is None else float(r)
    tgt = target_words(question, r)
    parts, seen, n = [], set(), 0
    for ri in _ordered_representatives(row_idxs):
        pk = CH.parent_key.values[ri]
        if pk in seen:
            continue
        seen.add(pk)
        head = _head(ri, head_mode)
        budget = max(60, tgt - n - len(head.split()) - (1 if head else 0))
        body = _anchor_body(ri, question, budget) if expand else CH_TEXT[ri]
        blk = f"{head}\n{body.strip()}" if head else body.strip()
        parts.append(blk)
        n += len(blk.split())
        if n >= tgt:
            break
    if not parts:
        return ""
    txt = "\n".join(parts)
    return _trim(txt, tgt) if n > tgt * 1.10 else txt


[built] len_table.json  0.3s  0.0MiB
qtype              n   median gold words
khac            3432                 306
ho_so            994                 348
trach_nhiem      746                 323
dieu_kien        418                 333
thoi_han         372                 299
xu_phat          337                 354
dinh_nghia       263                 230
tham_quyen       205                 294
muc_huong        133                 294
mau_bieu         100                  78


## S07b · Ba mốc sàn (không cần GPU)

Đo trước khi làm bất cứ thứ gì có GPU. Ba con số này định hình mọi quyết định còn lại:

| Mốc | Ý nghĩa |
|---|---|
| **scaffold** | chỉ template + boilerplate pháp lý, **không** retrieval → sàn tuyệt đối |
| **kNN answer-bank** | trả nguyên answer của train question gần nhất → sàn không cần corpus |
| **BM25 top-1** | `assemble()` trên chunk BM25 top-1 → sàn của nhánh extractive |

Nếu BM25 top-1 chưa vượt rõ kNN thì lỗi ở chunking/assembly, **không** phải ở model — và thêm GPU sẽ không cứu.

In [12]:
SCAFFOLD = ("Căn cứ quy định của pháp luật hiện hành, đối với trường hợp này được quy định như sau:\n"
            "1. Về nguyên tắc chung, cơ quan, tổ chức, cá nhân có liên quan phải tuân thủ các quy định "
            "tại Điều, khoản của văn bản quy phạm pháp luật có hiệu lực.\n"
            "2. Về trách nhiệm, các bên có trách nhiệm thực hiện đầy đủ nghĩa vụ theo quy định.\n"
            "- Trường hợp không thực hiện đúng quy định thì tuỳ theo tính chất, mức độ vi phạm sẽ bị "
            "xử lý theo quy định của pháp luật.\n"
            "- Đối với các nội dung khác không quy định tại văn bản này thì thực hiện theo quy định "
            "của pháp luật có liên quan.\n"
            "Theo đó, việc áp dụng phải căn cứ vào quy định cụ thể nêu trên.")

def eval_preds(pred: dict, ids) -> dict:
    m = np.array([meteor_1(TRAIN[i]["answer"], pred[i]) for i in ids])
    r = np.array([rougeL_1(TRAIN[i]["answer"], pred[i]) for i in ids])
    return dict(meteor=float(m.mean()), rouge=float(r.mean()),
                meteor_p10=float(np.percentile(m, 10)), meteor_p90=float(np.percentile(m, 90)))

if RUN["S07_floor"]:
    ids = EVAL_IDS[:600]

    # ---- floor 1: scaffold, quét theo r
    print("floor 1 · scaffold")
    for r in [1.0, 1.5, 2.0, 3.0]:
        pred = {}
        for i in ids:
            tgt = target_words(TRAIN[i]["question"], r)
            reps = max(1, math.ceil(tgt / len(SCAFFOLD.split())))
            pred[i] = _trim("\n".join([SCAFFOLD] * reps), tgt)
        print("   r=%.1f  %s" % (r, eval_preds(pred, ids)))

    # ---- floor 2: kNN answer-bank
    print("floor 2 · kNN answer-bank")
    pool = [i for i in TRAIN_IDS if i not in set(ids)]
    vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), sublinear_tf=True, min_df=2)
    Xp = vec.fit_transform([TRAIN[i]["question"] for i in pool])
    Xq = vec.transform([TRAIN[i]["question"] for i in ids])
    from sklearn.preprocessing import normalize
    S = (normalize(Xq) @ normalize(Xp).T).toarray()
    nn = S.argmax(1)
    pred = {i: TRAIN[pool[nn[k]]]["answer"] for k, i in enumerate(ids)}
    print("   %s" % eval_preds(pred, ids))

    # ---- floor 3: BM25 top-1 + assemble
    print("floor 3 · BM25 top-1 + assemble")
    top, _ = bm25_query(BM, [TRAIN[i]["question"] for i in ids], 5)
    for r in [1.0, 2.0, 2.5]:
        pred = {i: assemble([top[k, 0]], TRAIN[i]["question"], r) for k, i in enumerate(ids)}
        print("   r=%.1f  %s" % (r, eval_preds(pred, ids)))

    print("floor 3b · BM25 top-3 + assemble (r=2.0)")
    pred = {i: assemble(list(top[k, :3]), TRAIN[i]["question"], 2.0) for k, i in enumerate(ids)}
    print("   %s" % eval_preds(pred, ids))

floor 1 · scaffold
   r=1.0  {'meteor': 0.14053449395469972, 'rouge': 0.3086859385276437, 'meteor_p10': 0.09289950483381969, 'meteor_p90': 0.19120324516353418}
   r=1.5  {'meteor': 0.1442099515930385, 'rouge': 0.3028641976977597, 'meteor_p10': 0.09787338391519623, 'meteor_p90': 0.1938090961638812}
   r=2.0  {'meteor': 0.14060489840696946, 'rouge': 0.28796508459152886, 'meteor_p10': 0.09497285645007228, 'meteor_p90': 0.18302278404203096}
   r=3.0  {'meteor': 0.1328978186746759, 'rouge': 0.2579849543900817, 'meteor_p10': 0.0892904964169726, 'meteor_p90': 0.17769412548691246}
floor 2 · kNN answer-bank
   {'meteor': 0.28423211518152597, 'rouge': 0.37462202987685894, 'meteor_p10': 0.11250518794592435, 'meteor_p90': 0.517995008579481}
floor 3 · BM25 top-1 + assemble
   r=1.0  {'meteor': 0.34947671156254967, 'rouge': 0.4339867010342376, 'meteor_p10': 0.108631702455498, 'meteor_p90': 0.6855659161091416}
   r=2.0  {'meteor': 0.4008245415657904, 'rouge': 0.45607006415567847, 'meteor_p10': 0.1497

## S08 · Dense encoding trên 2 GPU

Hai thread, mỗi thread một bản model trên một device. GIL được nhả trong lúc CUDA chạy nên
pattern này đạt gần 2× mà không cần DDP/multiprocessing (rất dễ vỡ trong notebook).

Chunk sort theo độ dài trước khi batch → giảm padding, nhanh thêm ~30%.
`EMB_MAX_LEN=512` cho kênh recall; reranker mới dùng 1536 (chính khả năng 8k của bge-m3 cho phép điều đó —
các model PhoBERT-based chặn ở 258 token nên không dùng được cho việc này).

In [13]:
import torch.nn.functional as F
from concurrent.futures import ThreadPoolExecutor
from transformers import AutoTokenizer, AutoModel

def _encode_shard(device, texts, max_len, batch, is_query=False):
    tok = AutoTokenizer.from_pretrained(M_EMB)
    mdl = AutoModel.from_pretrained(M_EMB, torch_dtype=AMP_DTYPE).to(device).eval()
    order = np.argsort([len(t) for t in texts])           # sort by length
    out = np.zeros((len(texts), mdl.config.hidden_size), dtype=np.float16)
    with torch.inference_mode():
        for s in range(0, len(order), batch):
            idx = order[s:s+batch]
            enc = tok([texts[i] for i in idx], padding=True, truncation=True,
                      max_length=max_len, return_tensors="pt").to(device)
            # XLM-R may keep some activations in fp32; autocast aligns them with fp16 weights.
            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=str(device).startswith("cuda")):
                h = mdl(**enc).last_hidden_state[:, 0]     # bge-m3 dense = CLS
            out[idx] = F.normalize(h, dim=-1).cpu().numpy().astype(np.float16)
            if s % (batch * 200) == 0:
                print(f"   [{device}] {s}/{len(order)}", flush=True)
    del mdl; torch.cuda.empty_cache()
    return out

def encode(texts, max_len, batch):
    if len(DEVICES) == 1:
        return _encode_shard(DEVICES[0], texts, max_len, batch)
    cuts = np.linspace(0, len(texts), len(DEVICES) + 1).astype(int)
    shards = [(DEVICES[d], texts[cuts[d]:cuts[d+1]]) for d in range(len(DEVICES))]
    with ThreadPoolExecutor(max_workers=len(DEVICES)) as ex:
        res = list(ex.map(lambda a: _encode_shard(a[0], a[1], max_len, batch), shards))
    return np.concatenate(res, 0)

if RUN["S08_dense"]:
    with T(f"encode {len(CH_TEXT)} chunks"):
        EMB_C = cache("emb_chunks", lambda: encode(CH_TEXT, CFG["EMB_MAX_LEN"], CFG["EMB_BATCH"]), "npy")
    with T("encode queries"):
        ALLQ = [TRAIN[i]["question"] for i in TRAIN_IDS] + [PUBLIC[i]["question"] for i in PUBLIC_IDS]
        EMB_Q = cache("emb_queries", lambda: encode(ALLQ, 128, 128), "npy")
    QROW = {q: k for k, q in enumerate(TRAIN_IDS + PUBLIC_IDS)}
    print("emb_chunks", EMB_C.shape, "| emb_queries", EMB_Q.shape)

→ encode 252971 chunks ...


config.json:   0%|          | 0.00/664 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

   [cuda:0] 0/126485   [cuda:1] 0/126486

   [cuda:0] 6400/126485
   [cuda:1] 6400/126486
   [cuda:0] 12800/126485
   [cuda:1] 12800/126486
   [cuda:0] 19200/126485
   [cuda:1] 19200/126486
   [cuda:0] 25600/126485
   [cuda:1] 25600/126486
   [cuda:0] 32000/126485
   [cuda:1] 32000/126486
   [cuda:0] 38400/126485
   [cuda:1] 38400/126486
   [cuda:0] 44800/126485
   [cuda:1] 44800/126486
   [cuda:0] 51200/126485
   [cuda:1] 51200/126486
   [cuda:0] 57600/126485
   [cuda:1] 57600/126486
   [cuda:0] 64000/126485
   [cuda:1] 64000/126486
   [cuda:0] 70400/126485
   [cuda:1] 70400/126486
   [cuda:0] 76800/126485
   [cuda:1] 76800/126486
   [cuda:0] 83200/126485
   [cuda:1] 83200/126486
   [cuda:0] 89600/126485
   [cuda:1] 89600/126486
   [cuda:0] 96000/126485
   [cuda:1] 96000/126486
   [cuda:0] 102400/126485
   [cuda:1] 102400/126486
   [cuda:0] 108800/126485
   [cuda:1] 108800/126486
   [cuda:0] 115200/126485
   [cuda:1] 115200/126486
   [cuda:0] 121600/126485
   [cuda:1] 121600/126486
[b

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

   [cuda:0] 0/4000
   [cuda:1] 0/4000
[built] emb_queries.npy  11.5s  15.6MiB
← encode queries  11.5s
emb_chunks (252971, 1024) | emb_queries (8000, 1024)


## S09 · Retrieval: BM25 + dense + char rescorer → RRF + citation boost

RRF (`1/(60+rank)`) thay vì cộng điểm thô — bất biến với thang điểm của từng kênh, không cần tune trọng số.
Citation boost là **soft** (nhân điểm), không hard-filter: đề phòng parser bắt sai mã văn bản.

In [14]:
_EMB_C32 = None
def _chunk_mat():
    """float32 view của EMB_C, tạo MỘT lần. Không được .astype() trong vòng lặp:
    125k x 1024 float32 = 512MiB mỗi lần gọi."""
    global _EMB_C32
    if _EMB_C32 is None:
        _EMB_C32 = np.ascontiguousarray(EMB_C, dtype=np.float32)
        print(f"   [dense] chunk matrix {_EMB_C32.nbytes/2**20:.0f}MiB cached")
    return _EMB_C32

def dense_query(qrows, topk):
    Q = _chunk_mat()
    out_i = np.zeros((len(qrows), topk), dtype=np.int64)
    out_s = np.zeros((len(qrows), topk), dtype=np.float32)
    B = 128
    for s in range(0, len(qrows), B):
        qv = EMB_Q[qrows[s:s+B]].astype(np.float32)
        S = qv @ Q.T
        t = np.argpartition(-S, topk, axis=1)[:, :topk]
        o = np.take_along_axis(S, t, 1).argsort(1)[:, ::-1]
        out_i[s:s+B] = np.take_along_axis(t, o, 1)
        out_s[s:s+B] = np.take_along_axis(np.take_along_axis(S, t, 1), o, 1)
    return out_i, out_s

_CHAR_VEC = None
def char_rescore(question, cand_rows):
    global _CHAR_VEC
    if _CHAR_VEC is None:
        _CHAR_VEC = TfidfVectorizer(analyzer="char_wb", ngram_range=CFG["CHAR_NGRAM"],
                                    max_features=CFG["CHAR_MAX_FEAT"], sublinear_tf=True)
        _CHAR_VEC.fit([CH_TEXT[i][:2000] for i in
                       np.random.default_rng(SEED).choice(len(CH_TEXT), 20000, replace=False)])
    from sklearn.preprocessing import normalize
    A = normalize(_CHAR_VEC.transform([question]))
    B = normalize(_CHAR_VEC.transform([CH_TEXT[r][:2000] for r in cand_rows]))
    return (A @ B.T).toarray().ravel()

def rrf(rank_lists, k):
    sc = {}
    for lst in rank_lists:
        for rank, row in enumerate(lst):
            sc[row] = sc.get(row, 0.0) + 1.0 / (k + rank + 1)
    return sc

def retrieve(questions, qrows, topk=None):
    topk = topk or CFG["TOP_FUSE"]
    sp_i, _ = bm25_query(BM, questions, CFG["TOP_SPARSE"])
    dn_i, _ = dense_query(np.asarray(qrows), CFG["TOP_DENSE"])
    out = np.zeros((len(questions), topk), dtype=np.int64)
    for n, q in enumerate(questions):
        lists = [list(sp_i[n]), list(dn_i[n])]
        if CFG["USE_CHAR_RESCORE"]:
            cands = list(dict.fromkeys(lists[0] + lists[1]))[:100]
            cs = char_rescore(q, cands)
            lists.append([cands[i] for i in np.argsort(-cs)])
        sc = rrf(lists, CFG["RRF_K"])
        cit = parse_cit(q)
        if cit["van_ban"] and cit["dieu"]:
            for vb in cit["van_ban"]:
                for dn in cit["dieu"]:
                    for r in VB_DIEU.get((vb, dn), []):
                        sc[r] = sc.get(r, 0.0) * CFG["CIT_BOOST"] + 1.0
        best = sorted(sc.items(), key=lambda kv: -kv[1])[:topk]
        rows = [b[0] for b in best]
        out[n, :len(rows)] = rows
        if len(rows) < topk: out[n, len(rows):] = rows[-1] if rows else 0
    return out

if RUN["S09_retrieve"]:
    def _cands_train():
        qs = [TRAIN[i]["question"] for i in TRAIN_IDS]
        qr = [QROW[i] for i in TRAIN_IDS]
        chunks_out = []
        B = 500
        for s in range(0, len(qs), B):
            chunks_out.append(retrieve(qs[s:s+B], qr[s:s+B]))
            print(f"   retrieve train {s+B}/{len(qs)}", flush=True)
        return np.concatenate(chunks_out, 0)
    CAND_TR = cache("cand_train_v3", _cands_train, "npy")

    def _cands_pub():
        qs = [PUBLIC[i]["question"] for i in PUBLIC_IDS]
        qr = [QROW[i] for i in PUBLIC_IDS]
        return retrieve(qs, qr)
    CAND_PB = cache("cand_public_v3", _cands_pub, "npy")
    print("cand_train", CAND_TR.shape, "| cand_public", CAND_PB.shape)
    TR_POS = {q: k for k, q in enumerate(TRAIN_IDS)}
    PB_POS = {q: k for k, q in enumerate(PUBLIC_IDS)}

   [dense] chunk matrix 988MiB cached
   retrieve train 500/7000
   retrieve train 1000/7000
   retrieve train 1500/7000
   retrieve train 2000/7000
   retrieve train 2500/7000
   retrieve train 3000/7000
   retrieve train 3500/7000
   retrieve train 4000/7000
   retrieve train 4500/7000
   retrieve train 5000/7000
   retrieve train 5500/7000
   retrieve train 6000/7000
   retrieve train 6500/7000
   retrieve train 7000/7000
[built] cand_train_v3.npy  136.1s  2.7MiB
[built] cand_public_v3.npy  17.7s  0.4MiB
cand_train (7000, 50) | cand_public (1000, 50)


## S10 · Reranker (cross-encoder) trên 2 GPU

In [15]:
from transformers import AutoModelForSequenceClassification

def _rerank_shard(device, pairs, ckpt, max_len, batch):
    tok = AutoTokenizer.from_pretrained(ckpt)
    mdl = AutoModelForSequenceClassification.from_pretrained(
        ckpt, num_labels=1, torch_dtype=AMP_DTYPE).to(device).eval()
    out = np.zeros(len(pairs), dtype=np.float32)
    order = np.argsort([len(b) for _, b in pairs])
    with torch.inference_mode():
        for s in range(0, len(order), batch):
            idx = order[s:s+batch]
            a = [pairs[i][0] for i in idx]; b = [pairs[i][1] for i in idx]
            enc = tok(a, b, padding=True, truncation=True, max_length=max_len,
                      return_tensors="pt").to(device)
            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=str(device).startswith("cuda")):
                logits = mdl(**enc).logits[:, 0]
            out[idx] = logits.float().cpu().numpy()
            if s % (batch * 400) == 0: print(f"   [{device}] {s}/{len(order)}", flush=True)
    del mdl; torch.cuda.empty_cache()
    return out

def rerank(pairs, ckpt=None, max_len=None, batch=None):
    ckpt = ckpt or M_RERANK
    max_len = max_len or CFG["RERANK_MAX_LEN"]; batch = batch or CFG["RERANK_BATCH"]
    if len(DEVICES) == 1:
        return _rerank_shard(DEVICES[0], pairs, ckpt, max_len, batch)
    cuts = np.linspace(0, len(pairs), len(DEVICES) + 1).astype(int)
    with ThreadPoolExecutor(max_workers=len(DEVICES)) as ex:
        res = list(ex.map(lambda d: _rerank_shard(DEVICES[d], pairs[cuts[d]:cuts[d+1]],
                                                  ckpt, max_len, batch), range(len(DEVICES))))
    return np.concatenate(res, 0)

def rerank_set(qids, src, cand, topn, ckpt=None):
    """-> (n, TOP_FUSE) điểm reranker cho từng ứng viên."""
    pairs, meta = [], []
    for qi in qids:
        k = (TR_POS if src is TRAIN else PB_POS)[qi]
        for r in cand[k, :topn]:
            pairs.append((src[qi]["question"], CH_TEXT[r])); meta.append((qi, int(r)))
    sc = rerank(pairs, ckpt=ckpt)
    d = {}
    for (qi, r), s in zip(meta, sc): d.setdefault(qi, {})[r] = float(s)
    return d

if RUN["S10_rerank"]:
    ids = EVAL_IDS[:600]
    with T(f"rerank {len(ids)}x{CFG['TOP_FUSE']} pairs (zero-shot)"):
        RR0 = cache("rr_zeroshot_eval_v3", lambda: rerank_set(ids, TRAIN, CAND_TR, CFG["TOP_FUSE"]))
    print("ví dụ điểm:", sorted(list(RR0[ids[0]].values()), reverse=True)[:5])

→ rerank 600x50 pairs (zero-shot) ...


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

   [cuda:0] 0/15000
   [cuda:1] 0/15000
   [cuda:0] 3200/15000
   [cuda:1] 3200/15000
   [cuda:1] 6400/15000
   [cuda:0] 6400/15000
   [cuda:1] 9600/15000
   [cuda:0] 9600/15000
   [cuda:0] 12800/15000
   [cuda:1] 12800/15000
[built] rr_zeroshot_eval_v3.pkl  294.5s  0.4MiB
← rerank 600x50 pairs (zero-shot)  294.5s
ví dụ điểm: [5.10546875, 1.3486328125, 0.3642578125, -0.0489501953125, -0.5517578125]


## S11 · ⚠️ HEADROOM — cell quyết định thứ tự finetune

Ba khoảng trống, cộng lại bằng đúng phần điểm đang mất:

| Khoảng | Thuộc về | Finetune gì |
|---|---|---|
| `1.0 − oracle@200` | template / assembly | LLM, rule S07 |
| `oracle@200 − oracle@50` | recall | **embedding** |
| `oracle@50 − reranked@1` | precision | **reranker** |

**Khoảng nào lớn nhất thì làm cái đó trước.** Prior của tôi là reranker, nhưng cell này có quyền phủ quyết.
Cụ thể: `recall-proxy` thấp (oracle@200 ≫ oracle@50) ⇒ đổi sang embedding-first.

In [16]:
def oracle_at(qids, cand, pos_map, k, r=None):
    """max METEOR đạt được nếu chọn đúng chunk tốt nhất trong top-k.
    pos_map: qid -> hàng trong `cand` (tách khỏi TR_POS để dùng được với mọi bảng ứng viên)."""
    best, arg = {}, {}
    for qi in qids:
        pos = pos_map[qi]; gold = TRAIN[qi]["answer"]; q = TRAIN[qi]["question"]
        bs, ba = -1.0, None
        for row in dict.fromkeys(cand[pos, :k]):
            s = meteor_1(gold, assemble([row], q, r))
            if s > bs: bs, ba = s, row
        best[qi], arg[qi] = bs, ba
    return best, arg

if RUN["S11_headroom"]:
    ids = EVAL_IDS[:400]          # oracle đắt -> mẫu nhỏ hơn, vẫn đủ để ra quyết định
    r = CFG["R_DEFAULT"]
    POS200 = {qi: k for k, qi in enumerate(ids)}      # CAND200 index theo thứ tự `ids`

    with T("retrieve top-200 (union rộng, xấp xỉ oracle@∞)"):
        CAND200 = cache("cand_train_200_v3",
            lambda: np.concatenate([retrieve([TRAIN[i]["question"] for i in ids[s:s+200]],
                                             [QROW[i] for i in ids[s:s+200]], topk=200)
                                    for s in range(0, len(ids), 200)], 0), "npy")
    with T("oracle@200 / oracle@50"):
        o200, _ = oracle_at(ids, CAND200, POS200, 200, r)
        o50,  _ = oracle_at(ids, CAND200, POS200, 50,  r)

    rr1 = {}
    for qi in ids:
        rows = sorted(RR0[qi].items(), key=lambda kv: -kv[1]) if qi in RR0 else []
        top = [rows[0][0]] if rows else [CAND_TR[TR_POS[qi], 0]]
        rr1[qi] = meteor_1(TRAIN[qi]["answer"], assemble(top, TRAIN[qi]["question"], r))

    O200 = np.mean([o200[i] for i in ids]); O50 = np.mean([o50[i] for i in ids])
    R1   = np.mean([rr1[i]  for i in ids])
    gaps = {"assembly (1-oracle@200)": 1.0 - O200,
            "recall (oracle@200-oracle@50)": O200 - O50,
            "precision (oracle@50-reranked@1)": O50 - R1}
    print(f"oracle@200 = {O200:.4f}\noracle@50  = {O50:.4f}\nreranked@1 = {R1:.4f}\n")
    for k, v in sorted(gaps.items(), key=lambda kv: -kv[1]):
        print(f"  {k:<38} {v:.4f}")
    winner = max(gaps, key=gaps.get)
    print(f"\n>>> BOTTLENECK: {winner}")
    print(">>> assembly  -> ưu tiên S07 rule + S18 generator")
    print(">>> recall    -> ưu tiên finetune EMBEDDING (đổi thứ tự, bỏ qua S14 trước)")
    print(">>> precision -> ưu tiên finetune RERANKER (S13+S14, đúng plan mặc định)")
    json.dump({k: float(v) for k, v in gaps.items()} | {"oracle200": float(O200),
              "oracle50": float(O50), "reranked1": float(R1)},
              open(ART / "headroom.json", "w"), indent=2)

→ retrieve top-200 (union rộng, xấp xỉ oracle@∞) ...
[built] cand_train_200_v3.npy  9.6s  0.6MiB
← retrieve top-200 (union rộng, xấp xỉ oracle@∞)  9.6s
→ oracle@200 / oracle@50 ...
← oracle@200 / oracle@50  1669.5s
oracle@200 = 0.6463
oracle@50  = 0.6386
reranked@1 = 0.4961

  assembly (1-oracle@200)                0.3537
  precision (oracle@50-reranked@1)       0.1425
  recall (oracle@200-oracle@50)          0.0077

>>> BOTTLENECK: assembly (1-oracle@200)
>>> assembly  -> ưu tiên S07 rule + S18 generator
>>> recall    -> ưu tiên finetune EMBEDDING (đổi thứ tự, bỏ qua S14 trước)
>>> precision -> ưu tiên finetune RERANKER (S13+S14, đúng plan mặc định)


## S12 · Quét `r*` (tỉ lệ độ dài)

Công thức dự đoán vùng ngọt `r ≈ 2–2.5`. Cell này xác nhận bằng số thật, và tách theo `qtype`
vì câu `"Mẫu ..."` và câu `"...bị xử phạt thế nào?"` có phân phối độ dài rất khác.

In [17]:
if RUN["S12_rsweep"]:
    ids = FIT_IDS[:min(CFG["RSWEEP_N"], len(FIT_IDS))]
    def top_rows(qi, n=3):
        if qi in RR0:
            return [x[0] for x in sorted(RR0[qi].items(), key=lambda kv: -kv[1])[:n]]
        return list(CAND_TR[TR_POS[qi], :n])
    res = {}
    for r in CFG["R_GRID"]:
        m = np.array([meteor_1(TRAIN[i]["answer"], assemble(top_rows(i), TRAIN[i]["question"], r))
                      for i in ids])
        res[r] = m.mean(); print(f"  r={r:<4} METEOR={m.mean():.4f}")
    R_STAR = max(res, key=res.get)
    print(f"\n>>> r* = {R_STAR}  (mặc định đang là {CFG['R_DEFAULT']})")

    print("\nr tốt nhất theo qtype:")
    by = {}
    for i in ids: by.setdefault(qtype(TRAIN[i]["question"]), []).append(i)
    R_BY_TYPE = {}
    for t, tids in sorted(by.items(), key=lambda kv: -len(kv[1])):
        if len(tids) < 20: continue
        best, bs = None, -1
        for r in CFG["R_GRID"]:
            s = np.mean([meteor_1(TRAIN[i]["answer"], assemble(top_rows(i), TRAIN[i]["question"], r))
                         for i in tids])
            if s > bs: bs, best = s, r
        R_BY_TYPE[t] = best
        print(f"  {t:<14} n={len(tids):<5} r*={best:<4} METEOR={bs:.4f}")
    R_BY_TYPE["__global__"] = R_STAR
    json.dump(R_BY_TYPE, open(ART / "r_by_type.json", "w"))
    CFG["R_DEFAULT"] = R_STAR

  r=0.8  METEOR=0.3878
  r=1.0  METEOR=0.4294
  r=1.3  METEOR=0.4826
  r=1.6  METEOR=0.4991
  r=2.0  METEOR=0.5081
  r=2.5  METEOR=0.5089
  r=3.0  METEOR=0.5086

>>> r* = 2.5  (mặc định đang là 2.0)

r tốt nhất theo qtype:
  khac           n=598   r*=3.0  METEOR=0.5095
  ho_so          n=160   r*=2.0  METEOR=0.5337
  trach_nhiem    n=130   r*=3.0  METEOR=0.5170
  dieu_kien      n=75    r*=2.5  METEOR=0.4983
  thoi_han       n=65    r*=2.5  METEOR=0.5322
  xu_phat        n=63    r*=2.0  METEOR=0.5022
  dinh_nghia     n=42    r*=2.0  METEOR=0.4768
  tham_quyen     n=33    r*=2.0  METEOR=0.5562


## S13 · Nhãn METEOR cho reranker — **nhãn trên đúng hàm mục tiêu**

Với mỗi cặp `(question, chunk)` ta tính được METEOR **thật** mà chunk đó đạt:

```
label(q, chunk) = METEOR( assemble([chunk], q, r*), gold(q) )
```

Tốt hơn silver-citation ở bốn điểm: không nhiễu parse; phủ 7000/7000 (không chỉ 6760); vẫn đúng khi gold
trích văn bản **khác** cái được citation (rất hay xảy ra với bản hợp nhất/sửa đổi); và phân biệt được
chunk "đúng luật nhưng phủ ít token".

`assemble()` dùng ở đây **phải y hệt** lúc inference — đó là toàn bộ lý do S07 là nguồn chân lý duy nhất.

In [18]:
from multiprocessing import Pool

def _label_one(args):
    qi, q, gold, rows, r = args
    return qi, [(int(row), float(meteor_1(gold, assemble([row], q, r)))) for row in rows]

if RUN["S13_labels"]:
    rng = np.random.default_rng(SEED)
    lab_ids = list(FIT_IDS) if CFG["LABEL_N_TRAIN"] <= 0 else list(rng.choice(FIT_IDS, size=min(CFG["LABEL_N_TRAIN"], len(FIT_IDS)), replace=False))
    K = CFG["LABEL_TOP_K"]
    jobs = [(qi, TRAIN[qi]["question"], TRAIN[qi]["answer"],
             list(dict.fromkeys(CAND_TR[TR_POS[qi], :K])), q_rstar(TRAIN[qi]["question"])) for qi in lab_ids]
    print(f"{len(jobs)} query × ~{K} chunk = ~{len(jobs)*K:,} lần gọi METEOR")

    def _build():
        nw = max(1, (os.cpu_count() or 2))
        out = {}
        with Pool(nw) as p:
            for n, (qi, lst) in enumerate(p.imap_unordered(_label_one, jobs, chunksize=16)):
                out[qi] = lst
                if n % 500 == 0: print(f"   {n}/{len(jobs)}", flush=True)
        return out
    with T("METEOR labels"):
        LABELS = cache("meteor_labels_v3", _build)

    v = np.array([s for lst in LABELS.values() for _, s in lst])
    mx = np.array([max(s for _, s in lst) for lst in LABELS.values()])
    print(f"labels {len(v):,} | mean {v.mean():.4f} | best-per-query mean {mx.mean():.4f}")

5952 query × ~40 chunk = ~238,080 lần gọi METEOR
→ METEOR labels ...
   0/5952
   500/5952
   1000/5952
   1500/5952
   2000/5952
   2500/5952
   3000/5952
   3500/5952
   4000/5952
   4500/5952
   5000/5952
   5500/5952
[built] meteor_labels_v3.pkl  1985.2s  3.6MiB
← METEOR labels  1985.2s
labels 238,080 | mean 0.2706 | best-per-query mean 0.6369


## S14 · Finetune reranker — listwise KL trên nhãn METEOR

Listwise (softmax trên nhóm) thay vì pointwise regression: khớp đúng việc reranker phải làm là **xếp hạng
tương đối**, và giữ được thông tin có mức độ ("chunk này 0.62, chunk kia 0.58") mà contrastive nhị phân
sẽ ném bỏ.

Bộ nhớ 16GB T4: group 8, `max_len=512`, gradient checkpointing, AMP fp16.
Params fp32 2.27GB + grads 2.27GB + Adam m/v 4.55GB ≈ 9.1GB, activations nhỏ nhờ checkpointing → vừa.
`DataParallel` cho 2 GPU (DDP trong notebook rất dễ vỡ; DataParallel kém hiệu quả hơn nhưng đủ cho 568M).

In [19]:
_HAVE_FT = (FT_DIR / "config.json").exists() and "reranker_ft" not in FORCE
if _HAVE_FT:
    print(f"[skip] đã có checkpoint tại {FT_DIR} -> KHÔNG train lại.")
    print("       (quan trọng với notebook biến thể B/C: chúng dùng cùng reranker,")
    print("        không guard chỗ này thì mỗi biến thể tốn lại ~4h GPU vô ích)")
    print("       Muốn train lại: FORCE.add('reranker_ft')")

if RUN["S14_finetune"] and not _HAVE_FT:
    from torch.utils.data import Dataset, DataLoader
    import torch.nn.functional as F
    from transformers import AutoTokenizer, AutoModelForSequenceClassification

    tok_ft = AutoTokenizer.from_pretrained(M_RERANK)
    G, T_ = CFG["FT_GROUP"], CFG["FT_TEMP"]

    class ListwiseDS(Dataset):
        def __init__(s, labels):
            s.g = []
            for qi, lst in labels.items():
                lst = sorted(lst, key=lambda kv: -kv[1])
                if len(lst) < G or lst[0][1] <= 0: continue
                pos = lst[0]
                negs = lst[1:]
                pick = [pos] + [negs[i] for i in
                        np.random.default_rng(int.from_bytes(hashlib.sha1(str(qi).encode("utf-8")).digest()[:4], "little")).choice(len(negs),
                        min(G-1, len(negs)), replace=False)]
                if len(pick) < G: continue
                s.g.append((TRAIN[qi]["question"], pick))
        def __len__(s): return len(s.g)
        def __getitem__(s, i):
            q, pick = s.g[i]
            return q, [CH_TEXT[r] for r, _ in pick], np.array([v for _, v in pick], dtype=np.float32)

    def collate(b):
        qs, ps, ys = [], [], []
        for q, texts, y in b:
            qs += [q]*len(texts); ps += texts; ys.append(y)
        enc = tok_ft(qs, ps, padding=True, truncation=True,
                     max_length=CFG["FT_MAXLEN"], return_tensors="pt")
        return enc, torch.tensor(np.stack(ys))

    ds = ListwiseDS(LABELS)
    print("groups:", len(ds))
    dl = DataLoader(ds, batch_size=max(1, len(DEVICES)), shuffle=True,
                    collate_fn=collate, num_workers=2, drop_last=True)

    mdl = AutoModelForSequenceClassification.from_pretrained(M_RERANK, num_labels=1)
    mdl.gradient_checkpointing_enable()
    mdl = mdl.to(DEVICES[0])
    if len(DEVICES) > 1: mdl = torch.nn.DataParallel(mdl, device_ids=list(range(len(DEVICES))))
    opt = torch.optim.AdamW(mdl.parameters(), lr=CFG["FT_LR"], weight_decay=0.01)
    scaler = torch.cuda.amp.GradScaler()
    steps = CFG["FT_EPOCHS"] * len(dl)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=CFG["FT_LR"], total_steps=steps,
                                                pct_start=0.1)
    mdl.train(); step = 0
    for ep in range(CFG["FT_EPOCHS"]):
        for enc, y in dl:
            enc = {k: v.to(DEVICES[0]) for k, v in enc.items()}
            y = y.to(DEVICES[0])
            with torch.autocast("cuda", dtype=AMP_DTYPE):
                logits = mdl(**enc).logits.view(y.shape[0], -1)
            tgt = torch.softmax(y / T_, dim=-1)
            loss = F.kl_div(torch.log_softmax(logits.float(), -1), tgt, reduction="batchmean")
            scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(mdl.parameters(), 1.0)
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
            step += 1
            if step % 25 == 0:
                print(f"  ep{ep} step {step}/{steps} loss {loss.item():.4f} "
                      f"mem {torch.cuda.max_memory_allocated()/2**30:.1f}GiB", flush=True)
    (mdl.module if hasattr(mdl, "module") else mdl).save_pretrained(FT_DIR)
    tok_ft.save_pretrained(FT_DIR)
    del mdl, opt; gc.collect(); torch.cuda.empty_cache()
    print("saved ->", FT_DIR)

    # so sánh zero-shot vs finetuned trên cùng tập eval
    ids = EVAL_IDS[:400]
    RR1 = cache("rr_ft_eval_v3", lambda: rerank_set(ids, TRAIN, CAND_TR, CFG["TOP_FUSE"], ckpt=str(FT_DIR)))
    def _m(rr):
        return np.mean([meteor_1(TRAIN[i]["answer"],
                assemble([max(rr[i], key=rr[i].get)], TRAIN[i]["question"]))
                for i in ids if i in rr])
    print(f"reranked@1 zero-shot {_m(RR0):.4f}  ->  finetuned {_m(RR1):.4f}")

groups: 5952


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

/tmp/ipykernel_23/3358031843.py:52: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


  ep0 step 25/2976 loss 1.9008 mem 10.6GiB
  ep0 step 50/2976 loss 2.9117 mem 10.6GiB
  ep0 step 75/2976 loss 0.2009 mem 10.6GiB
  ep0 step 100/2976 loss 1.9826 mem 10.6GiB
  ep0 step 125/2976 loss 0.0179 mem 10.6GiB
  ep0 step 150/2976 loss 2.6017 mem 10.6GiB
  ep0 step 175/2976 loss 0.0915 mem 10.6GiB
  ep0 step 200/2976 loss 0.0730 mem 10.6GiB
  ep0 step 225/2976 loss 0.3707 mem 10.6GiB
  ep0 step 250/2976 loss 0.3830 mem 10.6GiB
  ep0 step 275/2976 loss 0.0876 mem 10.6GiB
  ep0 step 300/2976 loss 0.3935 mem 10.6GiB
  ep0 step 325/2976 loss 0.1365 mem 10.6GiB
  ep0 step 350/2976 loss 0.1016 mem 10.6GiB
  ep0 step 375/2976 loss 0.0682 mem 10.6GiB
  ep0 step 400/2976 loss 0.9824 mem 10.6GiB
  ep0 step 425/2976 loss 0.6430 mem 10.6GiB
  ep0 step 450/2976 loss 0.0215 mem 10.6GiB
  ep0 step 475/2976 loss 0.2663 mem 10.6GiB
  ep0 step 500/2976 loss 0.1528 mem 10.6GiB
  ep0 step 525/2976 loss 0.1359 mem 10.6GiB
  ep0 step 550/2976 loss 0.3907 mem 10.6GiB
  ep0 step 575/2976 loss 0.3296 mem

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved -> /kaggle/working/reranker_ft_v3


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

   [cuda:0] 0/10000
   [cuda:1] 0/10000
   [cuda:1] 3200/10000
   [cuda:0] 3200/10000
   [cuda:1] 6400/10000
   [cuda:0] 6400/10000
   [cuda:1] 9600/10000
   [cuda:0] 9600/10000
[built] rr_ft_eval_v3.pkl  191.0s  0.3MiB
reranked@1 zero-shot 0.5074  ->  finetuned 0.5437


## S15–S16 · Candidate K=6 + Selector (LightGBM, **0 tham số**)

Selector chọn cây thay vì head neural **chính vì** luật 4B: nó là thành phần học được mà không tốn một
tham số nào trong ngân sách. Trong cuộc thi bị chặn tham số, dồn trí thông minh vào các khối 0-param
(BM25, regex citation, rule assembly, LightGBM) là arbitrage.

Selector làm **hai** việc: chọn candidate, **và** quyết định khi nào bật floor guard ở S17.

In [20]:
CAND_SPECS = [
    dict(name="top1_default", kind="ranked", n=1, r=None, expand=True, head="default"),
    dict(name="top1_notitle", kind="ranked", n=1, r=None, expand=True, head="notitle"),
    dict(name="top1_none", kind="ranked", n=1, r=None, expand=True, head="none"),
    dict(name="top1_theo", kind="ranked", n=1, r=None, expand=True, head="theo"),
    dict(name="top1_high", kind="ranked", n=1, r=None, rmul=1.25, expand=True, head="default"),
    dict(name="top1_noexp", kind="ranked", n=1, r=None, expand=False, head="none"),
    dict(name="top1_neighbors", kind="neighbors", n=3, r=None, expand=True, head="notitle"),
    dict(name="top2_ranked", kind="ranked", n=2, r=None, expand=True, head="notitle"),
    dict(name="top2_legal", kind="legal", n=2, r=None, expand=True, head="notitle"),
    dict(name="top2_same_doc", kind="same_doc", n=2, r=None, expand=True, head="notitle"),
    dict(name="top3_legal", kind="legal", n=3, r=None, expand=True, head="notitle"),
    dict(name="top3_high", kind="ranked", n=3, r=None, rmul=1.20, expand=True, head="none"),
]

def _unique_rows(rows):
    out, seen = [], set()
    for r in rows:
        r = int(r)
        pk = CH.parent_key.values[r]
        if pk not in seen:
            seen.add(pk)
            out.append(r)
    return out

def _doc_neighbors(row, radius=2):
    doc = CH.doc_id.values[int(row)]
    rows = np.flatnonzero(CH.doc_id.values == doc).tolist()
    rows.sort(key=lambda x: int(CH.chunk_id.values[x]))
    try:
        pos = rows.index(int(row))
    except ValueError:
        return [int(row)]
    return rows[max(0, pos-radius): min(len(rows), pos+radius+1)]

def _candidate_rows(ranked_rows, spec):
    ranked_rows = [int(r) for r in ranked_rows]
    kind, n = spec.get("kind", "ranked"), int(spec["n"])
    if not ranked_rows:
        return []
    if kind == "neighbors":
        rows = _doc_neighbors(ranked_rows[0], radius=2) + ranked_rows
    elif kind == "same_doc":
        doc = CH.doc_id.values[ranked_rows[0]]
        rows = [r for r in ranked_rows if CH.doc_id.values[r] == doc] + ranked_rows
    else:
        rows = ranked_rows
    rows = _unique_rows(rows)[:n]
    if kind == "legal" or kind == "neighbors":
        if len(rows) > 1 and len({CH.doc_id.values[r] for r in rows}) == 1:
            rows.sort(key=lambda r: int(CH.chunk_id.values[r]))
    return rows

def make_candidates(question, ranked_rows, scores):
    out = []
    rstar = q_rstar(question)
    for sp in CAND_SPECS:
        r = (float(sp["r"]) if sp.get("r") is not None else rstar) * sp.get("rmul", 1.0)
        crows = _candidate_rows(ranked_rows, sp)
        txt = assemble(crows, question, r, expand=sp["expand"], head_mode=sp.get("head", "default"))
        out.append((sp["name"], txt, r, len(crows), crows))
    return out

def feats(question, txt, r, n, ranked_rows, scores, candidate_rows=None, candidate_name=""):
    cq = parse_cit(question)
    ct = parse_cit(txt)
    s = sorted(scores, reverse=True) + [0.0, 0.0]
    w = len(txt.split())
    tgt = target_words(question, r)
    toks = txt.split()
    uniq = len(set(toks)) / max(1, len(toks))
    crows = [int(x) for x in (candidate_rows or ranked_rows[:max(1, n)])]
    docs = {CH.doc_id.values[x] for x in crows}
    parents = {CH.parent_key.values[x] for x in crows}
    rank_map = {int(x): j for j, x in enumerate(ranked_rows)}
    ranks = [rank_map.get(x, len(ranked_rows)) for x in crows]
    adjacent = float(len(crows) > 1 and len(docs) == 1 and max(crows) - min(crows) <= 3)
    legal_order = float(len(crows) > 1 and crows == sorted(crows))
    return [s[0], s[1], s[0]-s[1], float(np.mean(s[:5])), r, n,
            w, w/max(1,tgt), len(question.split()),
            float(bool(cq["van_ban"])), float(bool(cq["dieu"])),
            float(len(set(cq["van_ban"]) & set(ct["van_ban"])) > 0),
            float(len(set(cq["dieu"]) & set(ct["dieu"])) > 0),
            len(ct["van_ban"]), len(ct["dieu"]), uniq,
            float(txt.count("\n")), float(len(re.findall(r"\d", txt))/max(1,w)),
            float(len(docs) == 1), float(len(parents)), float(np.mean(ranks)),
            adjacent, legal_order,
            float("notitle" in candidate_name), float("none" in candidate_name),
            float("neighbor" in candidate_name), float("legal" in candidate_name),
            float("high" in candidate_name)] \
        + [1.0 if qtype(question) == t else 0.0 for t, _ in QTYPE_RULES]

FEAT_NAMES = ["s1","s2","margin","s_mean5","r","n_span","words","words_over_tgt","q_words",
              "q_has_vb","q_has_dieu","vb_match","dieu_match","n_vb","n_dieu","uniq_ratio",
              "n_lines","digit_ratio","same_doc","n_parents","mean_rank","adjacent","legal_order",
              "head_notitle","head_none","neighbor","legal_candidate","high_length"] \
             + [f"qt_{t}" for t, _ in QTYPE_RULES]

def _build_selector_data(qids, rr_use):
    X, Y, Q, S = [], [], [], []
    for qi in qids:
        if qi not in rr_use:
            continue
        rr = sorted(rr_use[qi].items(), key=lambda kv: -kv[1])
        rows = [a for a, _ in rr]
        scs = [b for _, b in rr]
        q, gold = TRAIN[qi]["question"], TRAIN[qi]["answer"]
        for nm, txt, r, n, crows in make_candidates(q, rows, scs):
            if not txt:
                continue
            X.append(feats(q, txt, r, n, rows, scs, crows, nm))
            Y.append(meteor_1(gold, txt))
            Q.append(qi)
            S.append(nm)
    return dict(X=np.array(X, dtype=np.float32), Y=np.array(Y, dtype=np.float32),
                Q=np.array(Q), S=np.array(S))

def _selector_frame(X):
    return pd.DataFrame(X, columns=FEAT_NAMES)

def _new_selector(seed=SEED):
    try:
        import lightgbm as lgb
        return lgb.LGBMRegressor(n_estimators=700, learning_rate=0.04, num_leaves=31,
                                 min_child_samples=40, subsample=0.9,
                                 colsample_bytree=0.9, random_state=seed, verbose=-1)
    except ImportError:
        from sklearn.ensemble import HistGradientBoostingRegressor
        return HistGradientBoostingRegressor(max_iter=500, random_state=seed)

def _fit_selector_models(data, seeds):
    models = []
    Xf = _selector_frame(data["X"])
    for sd in seeds:
        m = _new_selector(sd)
        m.fit(Xf, data["Y"])
        models.append(m)
    return models

def _predict_selector(models, X):
    xf = _selector_frame(X)
    return np.mean([m.predict(xf) for m in models], axis=0)

def _pick_mean_pred(data, p):
    vals = []
    for qi in np.unique(data["Q"]):
        ix = np.flatnonzero(data["Q"] == qi)
        vals.append(float(data["Y"][ix[np.argmax(p[ix])]]))
    return float(np.mean(vals)) if vals else float("nan")

def _pick_mean(data, models):
    return _pick_mean_pred(data, _predict_selector(models, data["X"]))

if RUN["S15_candidates"]:
    # One all-train rerank pass is reused for fit/dev/all selector datasets.
    RR_SEL_ALL = cache("rr_ft_sel_v3", lambda: rerank_set(
        TRAIN_IDS, TRAIN, CAND_TR, CFG["TOP_FUSE"],
        ckpt=str(FT_DIR) if (FT_DIR/"config.json").exists() else None))
    SEL_FIT = cache("selector_fit_data_v3", lambda: _build_selector_data(FIT_IDS, RR_SEL_ALL))
    SEL_DEV = cache("selector_dev_data_v3", lambda: _build_selector_data(DEV_IDS, RR_SEL_ALL))
    SEL_ALL = cache("selector_all_data_v3", lambda: _build_selector_data(TRAIN_IDS, RR_SEL_ALL))
    print("selector fit", SEL_FIT["X"].shape, "dev", SEL_DEV["X"].shape, "all", SEL_ALL["X"].shape)

if RUN["S16_selector"]:
    groups = np.array([QGROUP[q] for q in SEL_FIT["Q"]])
    oof = np.zeros(len(SEL_FIT["Y"]), dtype=np.float32)
    for tr, te in GroupKFold(n_splits=5).split(SEL_FIT["X"], groups=groups):
        m = _new_selector(SEED)
        m.fit(_selector_frame(SEL_FIT["X"][tr]), SEL_FIT["Y"][tr])
        oof[te] = m.predict(_selector_frame(SEL_FIT["X"][te]))
    print("clean fit selector OOF picked:", round(_pick_mean_pred(SEL_FIT, oof), 4),
          "candidate oracle:", round(float(np.mean([
              np.max(SEL_FIT["Y"][SEL_FIT["Q"] == q]) for q in np.unique(SEL_FIT["Q"])])), 4))
    DEV_MODELS = _fit_selector_models(SEL_FIT, [SEED + i for i in range(3)])
    print("clean held-out DEV selector:", round(_pick_mean(SEL_DEV, DEV_MODELS), 4))
    # Deployment selector may use all official train answers.  No public gold
    # is involved; this is the final fit after the held-out diagnostic.
    SEL_MODELS = _fit_selector_models(SEL_ALL, [SEED + 100 + i for i in range(5)])
    pickle.dump(SEL_MODELS, open(ART / "selector_v3.pkl", "wb"))


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

   [cuda:1] 0/175000
   [cuda:0] 0/175000
   [cuda:0] 3200/175000
   [cuda:1] 3200/175000
   [cuda:0] 6400/175000
   [cuda:1] 6400/175000
   [cuda:1] 9600/175000
   [cuda:0] 9600/175000
   [cuda:1] 12800/175000
   [cuda:0] 12800/175000
   [cuda:0] 16000/175000
   [cuda:1] 16000/175000
   [cuda:0] 19200/175000
   [cuda:1] 19200/175000
   [cuda:0] 22400/175000
   [cuda:1] 22400/175000
   [cuda:0] 25600/175000
   [cuda:1] 25600/175000
   [cuda:0] 28800/175000
   [cuda:1] 28800/175000
   [cuda:0] 32000/175000
   [cuda:1] 32000/175000
   [cuda:0] 35200/175000
   [cuda:1] 35200/175000
   [cuda:0] 38400/175000
   [cuda:1] 38400/175000
   [cuda:0] 41600/175000
   [cuda:1] 41600/175000
   [cuda:0] 44800/175000
   [cuda:1] 44800/175000
   [cuda:0] 48000/175000
   [cuda:1] 48000/175000
   [cuda:0] 51200/175000
   [cuda:1] 51200/175000
   [cuda:0] 54400/175000
   [cuda:1] 54400/175000
   [cuda:0] 57600/175000
   [cuda:1] 57600/175000
   [cuda:0] 60800/175000
   [cuda:1] 60800/175000
   [cuda:0] 64

## S17 · Inference public + floor guard + validator + submission

Floor guard: điểm mấu chốt là METEOR final là **trung bình 1000 câu**, nên tiền nằm ở **đuôi dưới**
(kéo 100 câu từ 0.10→0.35 bằng đúng kéo 100 câu từ 0.55→0.80, nhưng dễ hơn nhiều lần). Vì vậy:
không bao giờ xuất chuỗi ngắn / rỗng / `"None"` / lời xin lỗi — kém nhất cũng phải là scaffold đúng độ dài.

Validator chặn cứng theo `scoring.py`: thiếu/thừa key ⇒ `KeyError` ⇒ **chết toàn bộ bài nộp**.

In [21]:
def predict_public():
    rr = cache("rr_public_v3", lambda: rerank_set(
        PUBLIC_IDS, PUBLIC, CAND_PB, CFG["TOP_FUSE"],
        ckpt=str(FT_DIR) if (FT_DIR/"config.json").exists() else None))
    out, n_dup, n_floor = {}, 0, 0
    for qi in PUBLIC_IDS:
        if qi in DUP_GATE:
            out[qi] = TRAIN[DUP_GATE[qi]]["answer"]
            n_dup += 1
            continue
        q = PUBLIC[qi]["question"]
        srt = sorted(rr[qi].items(), key=lambda kv: -kv[1])
        rows = [a for a, _ in srt]
        scs = [b for _, b in srt]
        cands = [(nm, t, r, n, crows) for nm, t, r, n, crows in make_candidates(q, rows, scs) if t]
        if cands and "SEL_MODELS" in globals():
            Xc = np.array([feats(q, t, r, n, rows, scs, crows, nm)
                           for nm, t, r, n, crows in cands], dtype=np.float32)
            pred = _predict_selector(SEL_MODELS, Xc)
            k = int(np.argmax(pred))
            txt, best_pred = cands[k][1], float(pred[k])
        elif cands:
            pick = next((c for c in cands if c[0] == "top2_legal"), cands[0])
            txt, best_pred = pick[1], 1.0
        else:
            txt, best_pred = "", -1.0
        if (not txt) or len(txt.split()) < 40 or best_pred < 0.03:
            tgt = target_words(q)
            base = assemble(rows[:3], q) or ""
            if len(base.split()) < 40:
                reps = max(1, math.ceil(tgt / len(SCAFFOLD.split())))
                base = _trim("\n".join([SCAFFOLD] * reps), tgt)
            txt = base
            n_floor += 1
        out[qi] = txt
    print(f"dup gate {n_dup} | floor guard {n_floor}")
    return out

def validate_and_pack(pred: dict, tag="v3"):
    # Codabench yêu cầu ZIP có đúng một submission.json ở root.
    expected = set(PUBLIC_IDS)
    actual = set(pred)
    assert actual == expected, (
        f"key lệch: thiếu {len(expected-actual)}, thừa {len(actual-expected)}"
    )
    sub = {}
    for k in PUBLIC_IDS:
        v = pred[k]
        assert isinstance(v, str) and v.strip() and v.strip().lower() != "none", (
            f"answer xấu tại {k}"
        )
        sub[k] = {"answer": v}

    SUB.mkdir(parents=True, exist_ok=True)
    p = SUB / "submission.json"
    with io.open(p, "w", encoding="utf-8") as f:
        json.dump(sub, f, ensure_ascii=False)
    import zipfile
    z = SUB / f"submission_{tag}.zip"
    with zipfile.ZipFile(z, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(p, arcname="submission.json")

    sha = hashlib.sha256(p.read_bytes()).hexdigest()
    w = np.array([len(v["answer"].split()) for v in sub.values()])
    man = dict(
        tag=tag, n=len(sub), sha256_json=sha,
        words=dict(mean=float(w.mean()), median=float(np.median(w)),
                   p10=float(np.percentile(w, 10)), max=int(w.max())),
        models=dict(emb=M_EMB, rerank=M_RERANK,
                     rerank_ft=str(FT_DIR) if (FT_DIR/"config.json").exists() else None,
                     gen=M_GEN if tag == VARIANT else None),
        cfg={k: v for k, v in CFG.items() if not isinstance(v, dict)},
        seed=SEED,
        headroom=json.load(open(ART/"headroom.json"))
                 if (ART/"headroom.json").exists() else None,
    )
    with io.open(SUB / f"manifest_{tag}.json", "w", encoding="utf-8") as f:
        json.dump(man, f, indent=2, ensure_ascii=False)
    print(json.dumps(man["words"], indent=2))
    print("sha256:", sha, "\nzip:", z, f"({z.stat().st_size/2**20:.1f} MiB)")
    return z

if RUN["S17_submit"]:
    with T("inference public v3"):
        PRED = predict_public()
    Z = validate_and_pack(PRED, tag="v3")
    k0 = PUBLIC_IDS[0]
    print("\n--- example ---\nQ:", PUBLIC[k0]["question"][:180])
    print("A:", PRED[k0][:700].replace("\n", " | "))


→ inference public v3 ...


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

   [cuda:0] 0/25000
   [cuda:1] 0/25000
   [cuda:1] 3200/25000
   [cuda:0] 3200/25000
   [cuda:1] 6400/25000
   [cuda:0] 6400/25000
   [cuda:1] 9600/25000
   [cuda:0] 9600/25000
   [cuda:1] 12800/25000
   [cuda:0] 12800/25000
   [cuda:1] 16000/25000
   [cuda:0] 16000/25000
   [cuda:1] 19200/25000
   [cuda:0] 19200/25000
   [cuda:1] 22400/25000
   [cuda:0] 22400/25000
[built] rr_public_v3.pkl  464.0s  0.7MiB
dup gate 1 | floor guard 0
← inference public v3  489.5s
{
  "mean": 558.293,
  "median": 537.0,
  "p10": 292.0,
  "max": 1154
}
sha256: 41ab1cb7ca4d4b9f10ac17c2d2f4672a0cfcaa6c1075f098cc1e37b14afaf1a4 
zip: /kaggle/working/submit/submission_v3.zip (0.8 MiB)

--- example ---
Q: Bộ quy tắc chuẩn mực đạo đức của doanh nghiệp xếp hạng tín nhiệm được áp dụng cho các đối tượng nào?
 |  | tham gia hợp đồng xếp hạng tín nhiệm theo quy định tại Khoản 2 Điều 38 Nghị định


## S18 · Tầng LLM của biến thể B

In [22]:
def copy_rate(text: str, context: str, n: int = 5) -> float:
    """Tỉ lệ n-gram của `text` xuất hiện NGUYÊN VĂN trong `context`.
    Proxy trực tiếp cho ch/m trong công thức METEOR = R/(0.9+0.1r) * (1 - 0.5(ch/m)^3):
      copy_rate ~ 1.0  -> khớp liền mạch  -> penalty ~ 0
      copy_rate ~ 0.3  -> paraphrase      -> penalty tới 50%
    Đây là thứ quyết định generator có dùng được hay không."""
    tt, ct = text.split(), context.split()
    if len(tt) < n: return 1.0
    ref = {" ".join(ct[i:i+n]) for i in range(max(0, len(ct)-n+1))}
    hit = sum(1 for i in range(len(tt)-n+1) if " ".join(tt[i:i+n]) in ref)
    return hit / max(1, len(tt)-n+1)

def ctx_of(rows, k=3) -> str:
    return "\n".join(_full_dieu(r) for r in rows[:k])

# sanity: extractive phải có copy_rate ~1.0
_r = int(CH.index[CH.dieu_no != ""][0])
_q = TRAIN[TRAIN_IDS[0]]["question"]
print("copy_rate(extractive) =", round(copy_rate(assemble([_r], _q, 2.0), ctx_of([_r])), 4))
print("copy_rate(scaffold)   =", round(copy_rate(SCAFFOLD, ctx_of([_r])), 4))

copy_rate(extractive) = 0.9623
copy_rate(scaffold)   = 0.0074


In [23]:
# Qwen3.5 selects the relevant legal spans.  It never writes the submitted
# answer directly; assemble() copies the selected source text verbatim.
from transformers import AutoProcessor, AutoModelForMultimodalLM

QWEN_TOPN = 8
QWEN_BATCH = 4
QWEN_MAX_INPUT = 3072
QWEN_MAX_NEW = 48
USE_QLL = False

QWEN_SYS = ("Bạn là bộ chọn căn cứ pháp luật Việt Nam. "
            "Chỉ chọn các mục cần thiết để trả lời câu hỏi. "
            "Không giải thích và không chép lại nội dung điều luật.")

def build_qwen_select_prompt(question, rows):
    options = "\n\n".join(
        f"[{j + 1}] {_head(r)}\n{_full_dieu(r)[:500]}"
        for j, r in enumerate(rows)
    )
    return (f"CÂU HỎI:\n{question}\n\nCÁC CĂN CỨ ỨNG VIÊN:\n{options}\n\n"
            "Trả lời đúng định dạng <selection>1,2</selection>. "
            "Chọn tối đa 3 số, theo mức độ liên quan giảm dần.")

def parse_qwen_selection(text, rows):
    match = re.search(r"<selection>\s*([^<]+)\s*</selection>", text, re.I)
    candidate = match.group(1) if match else text.strip().splitlines()[-1]
    nums = [int(x) for x in re.findall(r"\d+", candidate)]
    nums = [n for n in dict.fromkeys(nums) if 1 <= n <= len(rows)][:3]
    return [rows[n - 1] for n in nums] or rows[:2]

def _qwen_select_shard(device, jobs):
    processor = AutoProcessor.from_pretrained(M_GEN)
    model = AutoModelForMultimodalLM.from_pretrained(
        M_GEN, torch_dtype=AMP_DTYPE, low_cpu_mem_usage=True
    ).to(device).eval()
    selected = []
    for start in range(0, len(jobs), QWEN_BATCH):
        group = jobs[start:start + QWEN_BATCH]
        prompts = []
        for question, rows in group:
            messages = [
                {"role": "system", "content": [{"type": "text", "text": QWEN_SYS}]},
                {"role": "user", "content": [{"type": "text",
                                                    "text": build_qwen_select_prompt(question, rows)}]},
            ]
            try:
                prompt = processor.apply_chat_template(
                    messages, tokenize=False, add_generation_prompt=True,
                    enable_thinking=False)
            except TypeError:
                prompt = processor.apply_chat_template(
                    messages, tokenize=False, add_generation_prompt=True)
            prompts.append(prompt)
        inputs = processor(
            text=prompts, padding=True, truncation=True, max_length=QWEN_MAX_INPUT,
            return_tensors="pt").to(device)
        with torch.inference_mode():
            output_ids = model.generate(
                **inputs, max_new_tokens=QWEN_MAX_NEW, do_sample=False,
                pad_token_id=processor.tokenizer.pad_token_id or processor.tokenizer.eos_token_id)
        prompt_len = inputs["input_ids"].shape[1]
        for j, (_, rows) in enumerate(group):
            text = processor.decode(output_ids[j, prompt_len:], skip_special_tokens=True)
            selected.append(parse_qwen_selection(text, rows))
        if start % (QWEN_BATCH * 20) == 0:
            print(f"  [{device}] {start}/{len(jobs)}", flush=True)
    del model, processor
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return selected

def qwen_select_jobs(jobs):
    if not jobs:
        return []
    if len(DEVICES) == 1:
        return _qwen_select_shard(DEVICES[0], jobs)
    cuts = np.linspace(0, len(jobs), len(DEVICES) + 1).astype(int)
    with ThreadPoolExecutor(max_workers=len(DEVICES)) as executor:
        futures = [executor.submit(_qwen_select_shard, DEVICES[d], jobs[cuts[d]:cuts[d + 1]])
                   for d in range(len(DEVICES))]
        parts = [future.result() for future in futures]
    return [item for part in parts for item in part]


In [24]:
assert M_GEN == "Qwen/Qwen3.5-2B"
print("Qwen selector ready:", M_GEN)
print("mode: select up to 3 legal spans, then assemble verbatim")

Qwen selector ready: Qwen/Qwen3.5-2B
mode: select up to 3 legal spans, then assemble verbatim


In [25]:
# ------------------------------------------------------------------ S19  submission biến thể
# rr_public KHÔNG suffix theo biến thể: B và C dùng ĐÚNG embedding + reranker của A,
# nên ranking cross-encoder giống hệt -> tái dùng cache của A là đúng, không phải bug.
# Khác biệt giữa các biến thể nằm ở "ranking HIỆU DỤNG" sau tầng LLM, lưu riêng thành
# rr_eff_{VARIANT}.pkl để S20 ensemble được.
def _rr_public_base():
    return cache("rr_public_v3", lambda: rerank_set(
        PUBLIC_IDS, PUBLIC, CAND_PB, CFG["TOP_FUSE"],
        ckpt=str(FT_DIR) if (FT_DIR / "config.json").exists() else None))

In [26]:
def effective_ranking():
    rr = _rr_public_base()
    if globals().get("USE_QLL", False):
        print("ranking hiệu dụng = RRF(cross-encoder, query-likelihood)")
        fused, _ = cache(vname("qll_public"), lambda: qll_rerank(PUBLIC_IDS, PUBLIC, rr, 8))
        return fused
    print("ranking hiệu dụng = cross-encoder (QLL không cải thiện -> không dùng)")
    return rr

In [27]:
def predict_public_variant():
    rr = effective_ranking()
    work_ids, row_map, jobs = [], {}, []
    for qi in PUBLIC_IDS:
        if qi in DUP_GATE:
            continue
        rows = [int(r) for r, _ in sorted(rr[qi].items(), key=lambda kv: -kv[1])[:QWEN_TOPN]]
        work_ids.append(qi)
        row_map[qi] = rows
        jobs.append((PUBLIC[qi]["question"], rows))

    picked = cache(vname("qwen_select_public"), lambda: qwen_select_jobs(jobs))
    assert len(picked) == len(work_ids), "Qwen output count mismatch"
    picked_map = dict(zip(work_ids, picked))

    out, n_dup, n_floor = {}, 0, 0
    for qi in PUBLIC_IDS:
        if qi in DUP_GATE:
            out[qi] = TRAIN[DUP_GATE[qi]]["answer"]
            n_dup += 1
            continue
        q = PUBLIC[qi]["question"]
        txt = assemble(picked_map[qi], q) or ""
        if len(txt.split()) < 40:
            txt = assemble(row_map[qi][:3], q) or ""
            n_floor += 1
        if not txt:
            tgt = target_words(q)
            reps = max(1, math.ceil(tgt / len(SCAFFOLD.split())))
            txt = _trim("\n".join([SCAFFOLD] * reps), tgt)
            n_floor += 1
        out[qi] = txt
    print(f"Qwen={M_GEN} | selected {len(work_ids)} | dup gate {n_dup} | floor guard {n_floor}")
    return out

if RUN["S17_submit"] and RUN["S18_generator"]:
    with T(f"inference public — biến thể {VARIANT}"):
        PREDV = predict_public_variant()
    ZV = validate_and_pack(PREDV, tag=VARIANT)


→ inference public — biến thể B3_QWEN35_2B ...
[cache] rr_public_v3.pkl
ranking hiệu dụng = cross-encoder (QLL không cải thiện -> không dùng)


preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

[transformers] Qwen3_5ForConditionalGeneration LOAD REPORT from: Qwen/Qwen3.5-2B
Key            | Status  | 
---------------+---------+-
lm_head.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] Qwen3_5ForConditionalGeneration LOAD REPORT from: Qwen/Qwen3.5-2B
Key            | Status  | 
---------------+---------+-
lm_head.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


← inference public — biến thể B3_QWEN35_2B  50.3s


NotImplementedError: Cannot copy out of meta tensor; no data! Please use torch.nn.Module.to_empty() instead of torch.nn.Module.to() when moving module from meta to a different device.

In [ ]:
# v3 does not run an unvalidated ensemble.  Select the submission only
# after comparing the clean held-out DEV METEOR against the untouched B/v2
# baseline with the official local scorer.
print("v3 ensemble: disabled")


## Checklist trước khi nộp - pipeline v3

- [ ] Chạy `param_budget_variant()` và lưu số tham số thật của mọi model.
- [ ] Ba model trong pipeline đều có trong danh sách BTC đã phê duyệt.
- [ ] Không còn token/API/nguồn dữ liệu ngoài trong notebook.
- [ ] `DEV_IDS` và `FIT_IDS` tách theo `QGROUP`; chỉ chọn hyperparameter trên FIT.
- [ ] So sánh METEOR bằng đúng scorer BTC; không chọn theo ROUGE-L.
- [ ] `submission.zip` chứa duy nhất `submission.json` ở root, đủ 1.000 key.
- [ ] Không có answer rỗng/null/None; lưu manifest và checksum.
